[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jujumelona/material-candidate-explorer/blob/main/MATERIAL_CANDIDATE_DISCOVERY_T4.ipynb)

# Goal

Run a real, evidence-preserving material-candidate search on a Google Colab T4. Select one of 12 code-owned material-field routes, or use `AUTO` to reconcile deterministic evidence with an optional audited main-AI decision before generation. The model must return a typed field, confidence, subtype, and exact input evidence spans; conflicts, clarification requests, low confidence, and missing application context stop the run. The notebook never accepts a free-form model route or model-selected MCP tool. One adaptive, multi-round Fusion search maintains Pareto, stability, target-property, property-space-diversity (the internal branch identifier is `novelty`), and expert-disagreement branches under a global 8-32 candidate budget. External database novelty is assessed later and never inferred from the property-space branch. MatterSim and CHGNet re-evaluate every generated candidate as separate experts; the notebook never averages their energies or treats a generator target as a measured thermodynamic result.

Every scientific checkpoint uses the repository's `material-candidate-validation` operator skill, the typed field-and-stage-specific RAG/MCP router, and the stage's real runtime authority. The skill controls procedure; it is not a scientific predictor. Each checkpoint persists its bounded request, source/tool allowlist, raw source statuses, evidence handoff, official-validator state, artifact hashes, and failure interpretation. Missing or failed evidence stays `unknown` and never counts as a passing validator. Only source-closed `generation_prior` evidence may enter a Fusion decision context. Optional credentials are requested with hidden prompts and never written to the result archive. This generic T4 notebook generates crystals and performs MLIP triage; it does not calculate superconducting, electrochemical, catalytic, transport, magnetic, ferroelectric, alloy-service, or adsorption properties. Every required field property therefore remains explicitly `unknown` and external until its named high-fidelity workflow is actually executed.

In [ ]:
# @title 0. Search settings
DISCOVERY_PROMPT = "Find low-energy Li-O crystal candidates while preserving structural diversity." # @param {type:"string"}
MATERIAL_FIELD = "AUTO" # @param ["AUTO", "general_inorganic", "battery_electrode", "solid_electrolyte", "superconductor", "heterogeneous_catalyst", "semiconductor", "photovoltaic_absorber", "thermoelectric", "magnetic_material", "ferroelectric_piezoelectric", "structural_alloy", "porous_framework"]
MAIN_AI_FIELD_ROUTING = "AUTO" # @param ["AUTO", "REQUIRED", "OFF"]
MATERIAL_PROBLEM_CONTEXT_JSON = '{"target_property":"low_energy_structural_screening","temperature":300,"pressure":1.0}' # @param {type:"string"}
CHEMICAL_SYSTEM = "Li-O" # @param {type:"string"}
QUALITY = "MAXIMUM" # @param ["MAXIMUM"]
TOTAL_CANDIDATES = 16 # @param {type:"integer", min:8, max:32}
SEARCH_ROUNDS = 8 # @param {type:"integer", min:3, max:12}
MAX_GENERATION_CALLS = 32 # @param {type:"integer", min:8, max:128}
FRONTIER_WIDTH = 1 # @param {type:"integer", min:1, max:4}
DFT_TOP_K = 3 # @param {type:"integer", min:1, max:5}
BASE_SEED = 20260716 # @param {type:"integer"}
MINIMUM_DISTANCE_A = 0.70 # @param {type:"number"}
RELAX_STEPS = 150 # @param {type:"integer", min:1, max:2000}
RELAX_FMAX_EV_A = 0.05 # @param {type:"number"}
INSTALL_OR_REUSE_MODELS = True # @param {type:"boolean"}
DOWNLOAD_RESULTS_ZIP = True # @param {type:"boolean"}
ENABLE_STAGE_EVIDENCE = True # @param {type:"boolean"}
VALIDATION_EVIDENCE_MAX_RESULTS = 8 # @param {type:"integer", min:1, max:50}
VALIDATION_EVIDENCE_MAX_BRANCHES = 12 # @param {type:"integer", min:1, max:50}
VALIDATION_EVIDENCE_FROM_DATE = "" # @param {type:"string"}
LITERATURE_CONTACT_EMAIL = "" # @param {type:"string"}
RAG_MODEL_API_URL = "" # @param {type:"string"}
RAG_MODEL_NAME = "" # @param {type:"string"}
MATERIAL_FIELD_MODEL_API_URL = "" # @param {type:"string"}
MATERIAL_FIELD_MODEL_NAME = "" # @param {type:"string"}
MATERIAL_RAG_MCP_URL = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL_GENERATION_PRIOR = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL_IDENTITY_NOVELTY = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL_MLIP_DISAGREEMENT = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL_RELAXATION_VALIDATION = "" # @param {type:"string"}
MATERIAL_RAG_MCP_TOOL_DFT_HANDOFF = "" # @param {type:"string"}

import json
import re
from datetime import date

if not DISCOVERY_PROMPT.strip():
    raise ValueError("DISCOVERY_PROMPT is required.")
if MAIN_AI_FIELD_ROUTING not in {"AUTO", "REQUIRED", "OFF"}:
    raise ValueError("MAIN_AI_FIELD_ROUTING must be AUTO, REQUIRED, or OFF.")
if MAIN_AI_FIELD_ROUTING == "REQUIRED" and MATERIAL_FIELD.strip().upper() != "AUTO":
    raise ValueError("REQUIRED main-AI routing is valid only when MATERIAL_FIELD is AUTO.")
try:
    material_problem_context = json.loads(MATERIAL_PROBLEM_CONTEXT_JSON or "{}")
except json.JSONDecodeError as exc:
    raise ValueError("MATERIAL_PROBLEM_CONTEXT_JSON must be a JSON object.") from exc
if not isinstance(material_problem_context, dict):
    raise ValueError("MATERIAL_PROBLEM_CONTEXT_JSON must decode to a JSON object.")
context_secret_markers = ("api_key", "token", "secret", "password", "credential")
context_stack = [material_problem_context]
while context_stack:
    context_item = context_stack.pop()
    if isinstance(context_item, dict):
        if any(
            marker in str(key).casefold()
            for key in context_item
            for marker in context_secret_markers
        ):
            raise ValueError("MATERIAL_PROBLEM_CONTEXT_JSON cannot contain secret-like keys.")
        context_stack.extend(context_item.values())
    elif isinstance(context_item, list):
        context_stack.extend(context_item)
del context_stack, context_secret_markers
if QUALITY != "MAXIMUM":
    raise ValueError("This T4 workflow supports only the byte-attested MAXIMUM profile.")
if not 8 <= TOTAL_CANDIDATES <= 32:
    raise ValueError("TOTAL_CANDIDATES must be between 8 and 32.")
minimum_rounds_for_budget = 1 + (TOTAL_CANDIDATES - 1 + 4) // 5
if not minimum_rounds_for_budget <= SEARCH_ROUNDS <= 12:
    raise ValueError(
        f"SEARCH_ROUNDS must be {minimum_rounds_for_budget}-12 for this candidate budget."
    )
if not TOTAL_CANDIDATES <= MAX_GENERATION_CALLS <= 128:
    raise ValueError("MAX_GENERATION_CALLS must cover the candidate budget and be at most 128.")
if not 1 <= FRONTIER_WIDTH <= 4:
    raise ValueError("FRONTIER_WIDTH must be between 1 and 4.")
if not 1 <= DFT_TOP_K <= 5:
    raise ValueError("DFT_TOP_K must be between 1 and 5.")
if not 1 <= RELAX_STEPS <= 2000:
    raise ValueError("RELAX_STEPS must be between 1 and 2000.")
if MINIMUM_DISTANCE_A <= 0 or RELAX_FMAX_EV_A <= 0:
    raise ValueError("Geometry and relaxation thresholds must be positive.")
if not 1 <= VALIDATION_EVIDENCE_MAX_RESULTS <= 50:
    raise ValueError("VALIDATION_EVIDENCE_MAX_RESULTS must be between 1 and 50.")
if not 1 <= VALIDATION_EVIDENCE_MAX_BRANCHES <= 50:
    raise ValueError("VALIDATION_EVIDENCE_MAX_BRANCHES must be between 1 and 50.")
if VALIDATION_EVIDENCE_FROM_DATE.strip():
    date.fromisoformat(VALIDATION_EVIDENCE_FROM_DATE.strip())
if bool(RAG_MODEL_API_URL.strip()) != bool(RAG_MODEL_NAME.strip()):
    raise ValueError("RAG_MODEL_API_URL and RAG_MODEL_NAME must be set together.")
if bool(MATERIAL_FIELD_MODEL_API_URL.strip()) != bool(MATERIAL_FIELD_MODEL_NAME.strip()):
    raise ValueError(
        "MATERIAL_FIELD_MODEL_API_URL and MATERIAL_FIELD_MODEL_NAME must be set together."
    )
MCP_STAGE_TOOL_NAMES = {
    "generation_prior": MATERIAL_RAG_MCP_TOOL_GENERATION_PRIOR.strip(),
    "identity_novelty": MATERIAL_RAG_MCP_TOOL_IDENTITY_NOVELTY.strip(),
    "mlip_disagreement": MATERIAL_RAG_MCP_TOOL_MLIP_DISAGREEMENT.strip(),
    "relaxation_validation": MATERIAL_RAG_MCP_TOOL_RELAXATION_VALIDATION.strip(),
    "dft_handoff": MATERIAL_RAG_MCP_TOOL_DFT_HANDOFF.strip(),
}
configured_mcp_tools = {
    item for item in [MATERIAL_RAG_MCP_TOOL.strip(), *MCP_STAGE_TOOL_NAMES.values()]
    if item
}
if bool(MATERIAL_RAG_MCP_URL.strip()) != bool(configured_mcp_tools):
    raise ValueError(
        "Set MATERIAL_RAG_MCP_URL together with a fallback or stage-specific MCP tool."
    )
for tool_name in configured_mcp_tools:
    if re.fullmatch(r"[A-Za-z0-9_.-]{1,128}", tool_name) is None:
        raise ValueError("MCP tool names may contain only letters, digits, dots, hyphens, or underscores.")
STAGE_EVIDENCE_UNKNOWN_POLICY = "continue-with-official-validator-only-no-evidence-credit"

PERIODIC_TABLE = set(
    "H He Li Be B C N O F Ne Na Mg Al Si P S Cl Ar K Ca Sc Ti V Cr Mn Fe Co Ni "
    "Cu Zn Ga Ge As Se Br Kr Rb Sr Y Zr Nb Mo Tc Ru Rh Pd Ag Cd In Sn Sb Te I "
    "Xe Cs Ba La Ce Pr Nd Pm Sm Eu Gd Tb Dy Ho Er Tm Yb Lu Hf Ta W Re Os Ir Pt "
    "Au Hg Tl Pb Bi Po At Rn Fr Ra Ac Th Pa U Np Pu Am Cm Bk Cf Es Fm Md No Lr "
    "Rf Db Sg Bh Hs Mt Ds Rg Cn Nh Fl Mc Lv Ts Og".split()
)
symbols = [item for item in re.split(r"[-,\s]+", CHEMICAL_SYSTEM.strip()) if item]
if not 1 <= len(symbols) <= 16 or any(item not in PERIODIC_TABLE for item in symbols):
    raise ValueError("Use 1-16 element symbols, for example Li-O.")
chemical_system = "-".join(dict.fromkeys(symbols))
material_problem_context.setdefault("chemical_system", chemical_system)

BASE_GUIDANCE_ALPHA = 0.5
BASE_CFG_GAMMA = 2.0
GENERATION_BATCH_SIZE = 1
SEARCH_PLAN = {
    "quality": QUALITY,
    "requested_material_field": MATERIAL_FIELD,
    "main_ai_field_routing": MAIN_AI_FIELD_ROUTING,
    "material_problem_context_keys": sorted(material_problem_context),
    "rounds": SEARCH_ROUNDS,
    "frontier_width": FRONTIER_WIDTH,
    "max_generation_calls": MAX_GENERATION_CALLS,
    "max_generated_candidates": TOTAL_CANDIDATES,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "base_guidance_alpha": BASE_GUIDANCE_ALPHA,
    "base_classifier_free_guidance_gamma": BASE_CFG_GAMMA,
    "branches": ["pareto", "stability", "target_property", "novelty", "expert_disagreement"],
}
print(SEARCH_PLAN)


## Setup

Select Runtime > Change runtime type > T4 GPU, then run the next cell. Model environments stay isolated behind local HTTP sidecars. Materials Project and enabled evidence-service credentials are entered with hidden input and are never written to artifacts.

`MAIN_AI_FIELD_ROUTING=AUTO` uses `MATERIAL_FIELD_MODEL_API_URL`/`MATERIAL_FIELD_MODEL_NAME`, or reuses the configured RAG reasoning endpoint when the dedicated fields are blank, and otherwise falls back to deterministic routing. `REQUIRED` fails if no main model is configured; `OFF` uses only deterministic routing. The model returns a bounded enum decision with exact input evidence spans. Code reconciles that decision with deterministic evidence, stops on conflict or clarification, and never lets the model choose an endpoint, MCP tool, validator, or pass/fail result. Add all application conditions required by the intended field to `MATERIAL_PROBLEM_CONTEXT_JSON`; an incomplete route stops before generation.

`MATERIAL_RAG_MCP_TOOL` is an optional fallback. A stage-specific tool field overrides it for exactly one checkpoint. The endpoint and tool names come only from this setup cell; neither the discovery prompt nor a RAG model can choose them. Leave the endpoint and every MCP tool blank to skip MCP while Crossref/arXiv and any configured OpenAlex source continue.

In [ ]:
# @title 1. Install the public project and start isolated sidecars
from __future__ import annotations

import getpass
import hashlib
import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPOSITORY_URL = "https://github.com/jujumelona/material-candidate-explorer.git"
REPOSITORY_DIR = Path("/content/material-candidate-explorer")
RUN_ID = "material-t4-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = Path("/content") / RUN_ID
ARTIFACT_ROOT = RUN_ROOT / "artifacts"
AUDIT_CIF_ROOT = RUN_ROOT / "candidate-cifs-raw-audit"
UNIQUE_CIF_ROOT = RUN_ROOT / "crystallographically-unique-cifs"
RUN_ROOT.mkdir(parents=True, exist_ok=False)
ARTIFACT_ROOT.mkdir()
AUDIT_CIF_ROOT.mkdir()
UNIQUE_CIF_ROOT.mkdir()

def run_checked(command: list[str], *, cwd: Path | None = None, capture: bool = False):
    print("$", " ".join(shlex.quote(item) for item in command))
    return subprocess.run(
        command, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not (3, 11) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("The pinned materials extra requires Python 3.11-3.12.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a Colab T4 GPU runtime before continuing.")

if not (REPOSITORY_DIR / ".git").is_dir():
    run_checked(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)])
else:
    dirty = run_checked(
        ["git", "-C", str(REPOSITORY_DIR), "status", "--porcelain", "--untracked-files=no"],
        capture=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError("The public clone has tracked edits; restart Colab before updating.")
    run_checked(["git", "-C", str(REPOSITORY_DIR), "fetch", "--depth", "1", "origin", "main"])
    run_checked(["git", "-C", str(REPOSITORY_DIR), "checkout", "main"])
    run_checked(["git", "-C", str(REPOSITORY_DIR), "merge", "--ff-only", "origin/main"])

run_checked([
    sys.executable, "-m", "pip", "install", "-e", ".[materials]",
], cwd=REPOSITORY_DIR)

PROJECT_SKILL_ID = "material-candidate-validation"
PROJECT_SKILL_PATH = (
    REPOSITORY_DIR / ".codex" / "skills" / PROJECT_SKILL_ID / "SKILL.md"
)
if not PROJECT_SKILL_PATH.is_file():
    raise RuntimeError("The repository material-candidate-validation skill is missing.")
PROJECT_SKILL_SHA256 = hashlib.sha256(PROJECT_SKILL_PATH.read_bytes()).hexdigest()

os.environ.update({
    "MATTERGEN_PRETRAINED_NAME": "chemical_system_energy_above_hull",
    "MATTERGEN_DEVICE": "cuda",
    "MATTERSIM_DEVICE": "cpu",
    "CHGNET_DEVICE": "cpu",
})
if INSTALL_OR_REUSE_MODELS:
    run_checked([
        "bash", "bootstrap.sh", "--profile", "materials-open",
        "--accelerator", "cuda", "--include-weights",
    ], cwd=REPOSITORY_DIR)
    run_checked([
        "bash", "start-sidecars.sh", "--profile", "materials-open",
        "--component", "mattergen", "--component", "mattersim",
        "--component", "chgnet", "--ready-timeout-seconds", "900",
    ], cwd=REPOSITORY_DIR)

environment_file = REPOSITORY_DIR / ".discovery" / "sidecars.env.sh"
if not environment_file.is_file():
    raise RuntimeError("Sidecar environment was not created.")
raw_environment = run_checked([
    "bash", "-lc",
    "set -a; . " + shlex.quote(str(environment_file)) + "; env -0",
], capture=True).stdout.encode()
for entry in raw_environment.split(b"\0"):
    if b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

mp_key = getpass.getpass(
    "Materials Project API key (optional, hidden; press Enter to skip): "
).strip()
if mp_key:
    os.environ["MP_API_KEY"] = mp_key
del mp_key

stage_environment = {
    "VALIDATION_EVIDENCE_ENABLED": "1" if ENABLE_STAGE_EVIDENCE else "0",
    "VALIDATION_EVIDENCE_MAX_RESULTS": str(VALIDATION_EVIDENCE_MAX_RESULTS),
    "VALIDATION_EVIDENCE_MAX_BRANCHES": str(VALIDATION_EVIDENCE_MAX_BRANCHES),
    "VALIDATION_EVIDENCE_FROM_DATE": VALIDATION_EVIDENCE_FROM_DATE.strip(),
    "LITERATURE_CONTACT_EMAIL": LITERATURE_CONTACT_EMAIL.strip(),
    "RAG_MODEL_API_URL": RAG_MODEL_API_URL.strip(),
    "RAG_MODEL_NAME": RAG_MODEL_NAME.strip(),
    "MATERIAL_FIELD_MODEL_API_URL": MATERIAL_FIELD_MODEL_API_URL.strip(),
    "MATERIAL_FIELD_MODEL_NAME": MATERIAL_FIELD_MODEL_NAME.strip(),
    "MATERIAL_RAG_MCP_URL": MATERIAL_RAG_MCP_URL.strip(),
    "MATERIAL_RAG_MCP_TOOL": MATERIAL_RAG_MCP_TOOL.strip(),
    "MATERIAL_RAG_MCP_TOOL_GENERATION_PRIOR": MATERIAL_RAG_MCP_TOOL_GENERATION_PRIOR.strip(),
    "MATERIAL_RAG_MCP_TOOL_IDENTITY_NOVELTY": MATERIAL_RAG_MCP_TOOL_IDENTITY_NOVELTY.strip(),
    "MATERIAL_RAG_MCP_TOOL_MLIP_DISAGREEMENT": MATERIAL_RAG_MCP_TOOL_MLIP_DISAGREEMENT.strip(),
    "MATERIAL_RAG_MCP_TOOL_RELAXATION_VALIDATION": MATERIAL_RAG_MCP_TOOL_RELAXATION_VALIDATION.strip(),
    "MATERIAL_RAG_MCP_TOOL_DFT_HANDOFF": MATERIAL_RAG_MCP_TOOL_DFT_HANDOFF.strip(),
}
for key, value in stage_environment.items():
    if value:
        os.environ[key] = value

if ENABLE_STAGE_EVIDENCE or (
    MATERIAL_FIELD.strip().upper() == "AUTO" and MAIN_AI_FIELD_ROUTING != "OFF"
):
    hidden_prompts = [
        (
            "OPENALEX_API_KEY",
            "OpenAlex API key (hidden; press Enter to skip the OpenAlex source): ",
            ENABLE_STAGE_EVIDENCE,
        ),
        (
            "RAG_MODEL_API_KEY",
            "RAG model API key (optional, hidden; press Enter if not required): ",
            bool(RAG_MODEL_API_URL.strip()) and (
                ENABLE_STAGE_EVIDENCE
                or (
                    MATERIAL_FIELD.strip().upper() == "AUTO"
                    and MAIN_AI_FIELD_ROUTING != "OFF"
                )
            ),
        ),
        (
            "MATERIAL_FIELD_MODEL_API_KEY",
            "Main material-field model API key (optional, hidden; press Enter if not required): ",
            MATERIAL_FIELD.strip().upper() == "AUTO"
            and MAIN_AI_FIELD_ROUTING != "OFF"
            and bool(MATERIAL_FIELD_MODEL_API_URL.strip()),
        ),
        (
            "MATERIAL_RAG_MCP_TOKEN",
            "Material MCP bearer token (optional, hidden; press Enter if not required): ",
            ENABLE_STAGE_EVIDENCE and bool(MATERIAL_RAG_MCP_URL.strip()),
        ),
    ]
    for variable, prompt, required_by_configuration in hidden_prompts:
        if not required_by_configuration:
            continue
        secret_value = getpass.getpass(prompt).strip()
        if secret_value:
            os.environ[variable] = secret_value
        del secret_value
del stage_environment

import requests

health_requirements = {
    "mattergen": ("MATTERGEN_API_URL", {"generate"}),
    "mattersim": ("MATTERSIM_API_URL", {"features", "relax"}),
    "chgnet": ("CHGNET_API_URL", {"features", "relax"}),
}
EXPECTED_EVALUATOR_WEIGHT_REVISIONS = {
    "mattersim": "sha256:e3df9fa708725e3d453140646c7d1838324b347a3d1214cf1440522146f872b5",
    "chgnet": "sha256:d14ab7c0f093efe64b60a7bcd540bca10e74fb7f46c86108a079af60524659d1",
}
health_rows = []
for component, (variable, required_operations) in health_requirements.items():
    endpoint = os.environ.get(variable, "").rstrip("/")
    if not endpoint:
        raise RuntimeError(variable + " is missing.")
    response = requests.get(endpoint + "/health", timeout=30)
    response.raise_for_status()
    payload = response.json()
    operations = set(payload.get("operations", []))
    if not required_operations.issubset(operations):
        raise RuntimeError(
            component + " is missing operations: " +
            ", ".join(sorted(required_operations - operations))
        )
    weight_revision = payload.get("weight_revision")
    if component in EXPECTED_EVALUATOR_WEIGHT_REVISIONS and weight_revision != (
        EXPECTED_EVALUATOR_WEIGHT_REVISIONS[component]
    ):
        raise RuntimeError(component + " did not load the MAXIMUM manifest-pinned checkpoint bytes.")
    health_rows.append({
        "component": component,
        "status": payload.get("status"),
        "operations": sorted(operations),
        "weight_revision": weight_revision,
    })

print(json.dumps(health_rows, indent=2))
print("Run root:", RUN_ROOT)


## Steps

1. Build one auditable search goal with an immutable chemical-system constraint and a MatterGen baseline at alpha 0.5 (classifier-free-guidance gamma 2). 2. Route a source-closed generation-prior request and pass its persisted RAG bundle only when the typed handoff permits generation steering. 3. Run the repository's global-budgeted `fusion-search` for at least three rounds while preserving Pareto, stability, target-property, property-space-diversity (internal identifier `novelty`), and expert-disagreement branches; persist and verify the actually applied alpha, gamma, and target matrix. 4. Reconstruct every generated candidate and its byte-attested MatterSim + CHGNet evidence from the persisted search report and Evidence Store. 5. Parse the authoritative Candidate CIF, enforce cross-round strict uniqueness, relax both experts independently, and rank only within reduced composition. 6. Run external novelty lookup and all stage-specific evidence checkpoints. 7. Create 1-5 non-executed DFT input packages for the validated shortlist, reserving at most one slot for a completed strict external-database no-match while giving unknown results no novelty credit. The one-candidate per-call batch is intentional: it guarantees at least three real adaptive rounds even at the minimum global budget of eight, at the cost of more sidecar calls. Unknown evidence remains unknown; prepared DFT input is never reported as an executed calculation.

In [ ]:
# @title 2. Build one adaptive search goal and source-closed generation prior
from discovery_os.fusion_schemas import (
    GenerationControls,
    WorkspaceMode,
    WorkspaceRunConfig,
)
from discovery_os.hashing import candidate_content_hash, stable_hash
from discovery_os.integration_manifest import load_integration_manifest
from discovery_os.material_domains import (
    build_main_model_material_field_classifier_from_environment,
    build_material_domain_plan,
)
from discovery_os.materials_screening import GenerationConditions
from discovery_os.profiles import get_validation_profile
from discovery_os.validation_evidence import (
    ValidationEvidenceRequest,
    ValidationEvidenceStage,
    build_validation_evidence_router_from_environment,
)
from discovery_os.schemas import (
    Candidate,
    CandidateRef,
    CandidateRepresentation,
    CandidateType,
    DiscoveryDomain,
    DiscoveryGoal,
    GoalConstraint,
    ObjectiveDirection,
    PropertyObjective,
    RepresentationKind,
)

def ref_key(reference: CandidateRef) -> tuple[str, int, str]:
    return (
        reference.candidate_id,
        reference.version,
        reference.content_hash,
    )

STAGE_EXECUTION_CONTRACTS = {
    ValidationEvidenceStage.GENERATION_PRIOR: {
        "operator_skill_stage": "generation_prior",
        "allowed_literature_sources": ["crossref", "arxiv", "openalex", "mcp"],
        "official_validators": [
            "mattergen-supported-condition-allowlist",
            "evidence-driven-fusion-controller",
        ],
        "handoff_consumer": "discovery_os.fusion_schemas.FusionDecisionContext",
    },
    ValidationEvidenceStage.IDENTITY_NOVELTY: {
        "operator_skill_stage": "identity_novelty",
        "allowed_literature_sources": ["crossref", "arxiv", "openalex", "mcp"],
        "official_validators": [
            "pymatgen-structure-matcher",
            "materials-project-find-structure",
        ],
        "handoff_consumer": "discovery_os.novelty.StagedNoveltyAssessor",
    },
    ValidationEvidenceStage.MLIP_DISAGREEMENT: {
        "operator_skill_stage": "mlip_disagreement",
        "allowed_literature_sources": ["crossref", "arxiv", "mcp"],
        "official_validators": [
            "mattersim-sidecar",
            "chgnet-sidecar",
            "cross-model-unit-normalized-disagreement",
        ],
        "handoff_consumer": "discovery_os.materials_screening.classify_model_disagreement",
    },
    ValidationEvidenceStage.RELAXATION_VALIDATION: {
        "operator_skill_stage": "relaxation_validation",
        "allowed_literature_sources": ["crossref", "arxiv", "mcp"],
        "official_validators": [
            "ase-periodic-optimizer",
            "mattersim-relaxation",
            "chgnet-relaxation",
        ],
        "handoff_consumer": "discovery_os.relaxation.PeriodicRelaxationResult",
    },
    ValidationEvidenceStage.DFT_HANDOFF: {
        "operator_skill_stage": "dft_handoff",
        "allowed_literature_sources": ["crossref", "arxiv", "mcp"],
        "official_validators": [
            "periodic-dft-backend-contract",
            "external-pseudopotential-review",
            "reference-phase-convergence-review",
        ],
        "handoff_consumer": "discovery_os.dft_handoff.PeriodicDFTBackend",
    },
}
VALID_RUNTIME_AUTHORITY_STATES = {"completed", "partial", "unknown", "pending"}
stage_evidence_runs = {}
stage_checkpoint_receipts = {}
stage_checkpoint_receipt_paths = {}


def checkpoint_artifact_binding(path: Path | str) -> dict:
    resolved = Path(path).resolve()
    resolved.relative_to(RUN_ROOT.resolve())
    if not resolved.is_file():
        raise ValueError("Checkpoint artifact is missing: " + str(resolved))
    encoded = resolved.read_bytes()
    return {
        "relative_path": str(resolved.relative_to(RUN_ROOT.resolve())),
        "byte_size": len(encoded),
        "sha256": hashlib.sha256(encoded).hexdigest(),
    }


def validate_runtime_validator_state(stage, state: dict) -> None:
    expected = STAGE_EXECUTION_CONTRACTS[stage]["official_validators"]
    outcomes = state.get("authority_outcomes")
    if not isinstance(outcomes, dict) or set(outcomes) != set(expected):
        raise ValueError(str(stage) + " must report every official validator authority.")
    if any(value not in VALID_RUNTIME_AUTHORITY_STATES for value in outcomes.values()):
        raise ValueError(str(stage) + " has an invalid runtime validator state.")
    if not isinstance(state.get("summary"), dict):
        raise ValueError(str(stage) + " runtime validator state requires a summary.")
    if state.get("unknown_is_pass") is not False:
        raise ValueError("Runtime validator unknown must never be recorded as pass.")


def run_stage_evidence_checkpoint(
    *,
    request: ValidationEvidenceRequest,
    goal: DiscoveryGoal,
    runtime_validator_state: dict,
    validator_output_artifacts: list[Path | str],
):
    stage = request.stage
    stage_name = str(stage)
    if stage_name in stage_evidence_runs:
        raise ValueError("Stage evidence checkpoint already ran: " + stage_name)
    contract = STAGE_EXECUTION_CONTRACTS[stage]
    validate_runtime_validator_state(stage, runtime_validator_state)
    checkpoint_root = RUN_ROOT / "stage-checkpoints" / stage_name
    checkpoint_root.mkdir(parents=True, exist_ok=False)
    request_path = checkpoint_root / "evidence-request.json"
    request_path.write_text(request.model_dump_json(indent=2) + "\n", encoding="utf-8")

    run = stage_evidence_router.run(request, goal=goal)
    report = run.report
    route = report.route
    handoff = report.handoff
    if request.material_field != domain_plan.profile.material_field:
        raise AssertionError("Every checkpoint must use the selected material field.")
    if report.material_field != domain_plan.profile.material_field:
        raise AssertionError("Evidence report material field changed in transit.")
    if report.domain_route is None or report.domain_route.profile_id != domain_plan.profile.profile_id:
        raise AssertionError("Evidence report is missing the selected field-and-stage route.")
    if report.request_hash != stable_hash(request):
        raise AssertionError("Persisted stage request hash changed.")
    if [str(item) for item in route.literature_sources] != contract[
        "allowed_literature_sources"
    ]:
        raise AssertionError(stage_name + " literature source allowlist changed.")
    if list(route.official_validators) != contract["official_validators"]:
        raise AssertionError(stage_name + " official validator allowlist changed.")
    if route.handoff_contract.consumer != contract["handoff_consumer"]:
        raise AssertionError(stage_name + " handoff consumer changed.")
    if route.mcp_policy != "configured-tool-only":
        raise AssertionError("A prompt or model must not select an MCP tool.")
    if route.mcp_contract.selection_policy != "administrator-configured-allowlist-only":
        raise AssertionError("MCP tool selection must stay configuration-owned.")
    if any(authority.failure_policy != "unknown-not-pass" for authority in route.validator_authorities):
        raise AssertionError("Every scientific validator must fail closed.")
    actual_source_statuses = {str(item.source) for item in report.source_statuses}
    if not actual_source_statuses.issubset(set(contract["allowed_literature_sources"])):
        raise AssertionError("Evidence report contains a non-allowlisted source.")
    if handoff.property_score_created or not handoff.unknown_not_pass:
        raise AssertionError("Evidence handoff crossed the scientific decision boundary.")
    if handoff.can_steer_generation and not (
        stage == ValidationEvidenceStage.GENERATION_PRIOR
        and handoff.evidence_available
    ):
        raise AssertionError("Only available generation-prior evidence may steer generation.")
    if stage != ValidationEvidenceStage.GENERATION_PRIOR and handoff.can_steer_generation:
        raise AssertionError("Post-generation evidence cannot steer a generator.")
    status = str(report.status)
    if status in {"unknown", "skipped"}:
        if handoff.evidence_available or report.record_count:
            raise AssertionError("Unknown/skipped evidence cannot expose usable records.")
        continuation = "official-validator-only-evidence-unknown"
    elif status in {"completed", "partial"}:
        if not handoff.evidence_available or run.bundle is None:
            raise AssertionError("Usable stage evidence requires a persisted bundle.")
        continuation = "source-grounded-context-plus-official-validator"
    else:
        raise AssertionError("Unsupported stage evidence status: " + status)

    configured_tool = (
        os.environ.get(route.mcp_contract.tool_environment_variable, "").strip()
        or os.environ.get(route.mcp_contract.fallback_tool_environment_variable, "").strip()
        or None
    )
    if str(report.mcp_contract_status) == "verified" and report.mcp_tool_name != configured_tool:
        raise AssertionError("Verified MCP tool differs from the configured stage allowlist.")
    if str(report.mcp_contract_status) != "verified" and report.mcp_tool_name is not None:
        raise AssertionError("An unverified MCP tool cannot appear in provenance.")

    artifact_bindings = []
    seen_paths = set()
    for path in validator_output_artifacts:
        binding = checkpoint_artifact_binding(path)
        if binding["relative_path"] not in seen_paths:
            artifact_bindings.append(binding)
            seen_paths.add(binding["relative_path"])
    evidence_outputs = {
        "report": checkpoint_artifact_binding(run.report_path),
        "bundle": (
            checkpoint_artifact_binding(RUN_ROOT / report.bundle_relative_path)
            if report.bundle_relative_path else None
        ),
    }
    receipt = {
        "checkpoint_id": "material-validation-" + stage_name + "-v1",
        "stage": stage_name,
        "operator_skill": {
            "skill_id": PROJECT_SKILL_ID,
            "stage": contract["operator_skill_stage"],
            "relative_path": str(PROJECT_SKILL_PATH.relative_to(REPOSITORY_DIR)),
            "sha256": PROJECT_SKILL_SHA256,
            "role": "operator-procedure-not-scientific-validator",
        },
        "input": {
            "evidence_request": checkpoint_artifact_binding(request_path),
            "request_hash": report.request_hash,
            "prompt_hash": report.prompt_hash,
            "material_field": domain_plan.profile.material_field.value,
            "material_domain_profile_id": domain_plan.profile.profile_id,
            "candidate_refs": [item.model_dump(mode="json") for item in request.candidate_refs],
            "composition_keys": list(request.composition_keys),
        },
        "allowlist": {
            "literature_sources": contract["allowed_literature_sources"],
            "official_validators": contract["official_validators"],
            "mcp_policy": route.mcp_policy,
            "mcp_capability": str(route.mcp_contract.capability),
            "mcp_tool_environment_variable": route.mcp_contract.tool_environment_variable,
            "mcp_fallback_environment_variable": route.mcp_contract.fallback_tool_environment_variable,
            "configured_mcp_tool": configured_tool,
            "model_or_prompt_can_select_tool": False,
        },
        "runtime_validator": runtime_validator_state,
        "validator_output_artifacts": artifact_bindings,
        "evidence_output": {
            **evidence_outputs,
            "report_id": report.report_id,
            "bundle_id": report.bundle_id,
            "source_statuses": [item.model_dump(mode="json") for item in report.source_statuses],
            "mcp_contract_status": str(report.mcp_contract_status),
            "mcp_tool_name": report.mcp_tool_name,
            "handoff": handoff.model_dump(mode="json"),
        },
        "status_handling": {
            "evidence_status": status,
            "reason": report.reason,
            "continuation": continuation,
            "unknown_is_pass": False,
            "evidence_can_replace_validator": False,
            "property_score_created": False,
        },
    }
    receipt_path = checkpoint_root / "checkpoint-receipt.json"
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
    stage_evidence_runs[stage_name] = run
    stage_checkpoint_receipts[stage_name] = receipt
    stage_checkpoint_receipt_paths[stage_name] = receipt_path
    return run


def finalize_stage_runtime_validator(
    stage: ValidationEvidenceStage,
    *,
    runtime_validator_state: dict,
    validator_output_artifacts: list[Path | str],
) -> None:
    stage_name = str(stage)
    validate_runtime_validator_state(stage, runtime_validator_state)
    receipt = dict(stage_checkpoint_receipts[stage_name])
    receipt["runtime_validator"] = runtime_validator_state
    receipt["validator_output_artifacts"] = [
        checkpoint_artifact_binding(path) for path in validator_output_artifacts
    ]
    receipt_path = stage_checkpoint_receipt_paths[stage_name]
    receipt_path.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
    stage_checkpoint_receipts[stage_name] = receipt


def seed_cif(elements: list[str]) -> str:
    rows = []
    for index, element in enumerate(elements):
        x = (index % 4) / 4.0
        y = ((index // 4) % 4) / 4.0
        z = ((index // 16) % 4) / 4.0
        rows.append(f"{element}{index + 1} {element} {x:.6f} {y:.6f} {z:.6f} 1.0")
    return "\n".join([
        "data_material_search_seed",
        "_symmetry_space_group_name_H-M 'P 1'",
        "_symmetry_Int_Tables_number 1",
        "_cell_length_a 8.000",
        "_cell_length_b 8.000",
        "_cell_length_c 8.000",
        "_cell_angle_alpha 90.000",
        "_cell_angle_beta 90.000",
        "_cell_angle_gamma 90.000",
        "loop_",
        "_atom_site_label",
        "_atom_site_type_symbol",
        "_atom_site_fract_x",
        "_atom_site_fract_y",
        "_atom_site_fract_z",
        "_atom_site_occupancy",
        *rows,
        "",
    ])

main_model_run = None
main_ai_field_routing_status = "not-requested-explicit-material-field"
if MATERIAL_FIELD.strip().upper() == "AUTO":
    if MAIN_AI_FIELD_ROUTING == "OFF":
        main_ai_field_routing_status = "off-deterministic-fallback"
    else:
        main_field_classifier = (
            build_main_model_material_field_classifier_from_environment(
                required=MAIN_AI_FIELD_ROUTING == "REQUIRED"
            )
        )
        if main_field_classifier is None:
            main_ai_field_routing_status = "unconfigured-deterministic-fallback"
        else:
            main_model_run = main_field_classifier.classify(
                DISCOVERY_PROMPT,
                chemical_system=chemical_system,
                problem_context=material_problem_context,
            )
            main_ai_field_routing_status = "executed-audited-decision"

domain_plan = build_material_domain_plan(
    MATERIAL_FIELD,
    prompt=DISCOVERY_PROMPT,
    chemical_system=chemical_system,
    problem_context=material_problem_context,
    model_run=main_model_run,
)
if domain_plan.resolution.requires_operator_choice:
    ambiguous = ", ".join(
        item.value for item in domain_plan.resolution.ambiguous_fields
    )
    clarification = (
        domain_plan.main_model_run.decision.clarification_question
        if domain_plan.main_model_run is not None
        and domain_plan.main_model_run.decision.needs_clarification
        else None
    )
    raise ValueError(
        "AUTO material-field resolution requires operator clarification before generation. "
        + ("Conflicting fields: " + ambiguous + ". " if ambiguous else "")
        + ("Model question: " + clarification + " " if clarification else "")
        + "Choose MATERIAL_FIELD explicitly or refine the prompt/context."
    )
if not domain_plan.field_route_ready:
    raise ValueError(
        "The selected material-field route is missing required problem context before "
        "generation: " + ", ".join(domain_plan.missing_required_context)
        + ". Add the values to MATERIAL_PROBLEM_CONTEXT_JSON."
    )
selected_validation_profile = get_validation_profile(
    domain_plan.profile.discovery_domain
)
GENERIC_T4_FIELD_PROPERTY_BOUNDARY = (
    "This generic T4 run generates CRYSTAL candidates and performs MatterSim/CHGNet "
    "MLIP screening only. It does not execute the selected field's property-specific "
    "physics simulations or experiments; every unexecuted required field property "
    "remains unknown/external and receives no score or pass credit."
)
field_property_status = {
    property_name: {
        "status": "unknown",
        "execution": "not_executed",
        "authority": "external-required",
        "unknown_is_pass": False,
    }
    for property_name in domain_plan.unexecuted_required_properties
}
material_domain_audit = {
    "plan": domain_plan.model_dump(mode="json"),
    "resolution_mode": domain_plan.resolution.selection_mode,
    "resolution_confidence": domain_plan.resolution.confidence,
    "operator_choice_required": domain_plan.resolution.requires_operator_choice,
    "field_route_ready": domain_plan.field_route_ready,
    "missing_required_context": domain_plan.missing_required_context,
    "main_ai_field_routing_setting": MAIN_AI_FIELD_ROUTING,
    "main_ai_field_routing_status": main_ai_field_routing_status,
    "main_model_run": (
        domain_plan.main_model_run.model_dump(mode="json")
        if domain_plan.main_model_run is not None else None
    ),
    "accepted_route_input": "explicit-enum-or-audited-model-plus-deterministic-reconciliation",
    "broad_validation_profile_id": selected_validation_profile.profile_id,
    "main_model_decision_id": (
        domain_plan.main_model_run.decision_id
        if domain_plan.main_model_run is not None else None
    ),
    "candidate_type_used_by_t4": "crystal",
    "generic_mlip_screening": ["MatterSim", "CHGNet"],
    "field_property_status": field_property_status,
    "field_properties_calculated_by_t4": [],
    "scientific_boundary": GENERIC_T4_FIELD_PROPERTY_BOUNDARY,
}
MATERIAL_DOMAIN_PLAN_PATH = RUN_ROOT / "material-domain-plan.json"
MATERIAL_DOMAIN_PLAN_PATH.write_text(
    domain_plan.model_dump_json(indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps({
    "selected_material_field": domain_plan.profile.material_field.value,
    "field_profile_id": domain_plan.profile.profile_id,
    "broad_validation_profile_id": selected_validation_profile.profile_id,
    "main_ai_field_routing_status": main_ai_field_routing_status,
    "main_model_decision_id": (
        domain_plan.main_model_run.decision_id
        if domain_plan.main_model_run is not None else None
    ),
    "unexecuted_required_properties": domain_plan.unexecuted_required_properties,
}, indent=2))

draft_parent = Candidate(
    candidate_id="colab-material-lineage-seed",
    candidate_type=CandidateType.CRYSTAL,
    domain=domain_plan.profile.discovery_domain,
    name=chemical_system + " lineage seed",
    representations=[CandidateRepresentation(
        kind=RepresentationKind.CIF,
        value=seed_cif(symbols),
        media_type="chemical/x-cif",
        format_version="CIF 1.1",
        canonical=False,
    )],
    attributes={"chemical_system": chemical_system, "seed_only": True},
    provenance={"role": "lineage_seed_not_a_stability_claim"},
)
parent_data = draft_parent.model_dump(mode="python")
parent_data["candidate_ref"] = CandidateRef(
    candidate_id=draft_parent.candidate_id,
    version=1,
    content_hash=candidate_content_hash(draft_parent),
)
parent = Candidate.model_validate(parent_data)
PARENT_PATH = RUN_ROOT / "parent.json"
PARENT_PATH.write_text(parent.model_dump_json(indent=2), encoding="utf-8")

manifest = load_integration_manifest()
mattergen_component = next(
    item for item in manifest.components if item.component_id == "mattergen"
)
mattergen_weight_revision = os.environ["MATTERGEN_WEIGHT_REVISION"].strip()

base_conditions = GenerationConditions(
    profile_id="adaptive-search-baseline",
    guidance_alpha=BASE_GUIDANCE_ALPHA,
    target_energy_above_hull_eV_atom=None,
)
search_goal = DiscoveryGoal(
    goal_id="material-goal-adaptive-search",
    domain=domain_plan.profile.discovery_domain,
    title=chemical_system + " adaptive multi-round crystal search",
    scientific_question=DISCOVERY_PROMPT.strip(),
    objectives=[
        PropertyObjective(
            property_name="energy_per_atom",
            direction=ObjectiveDirection.MINIMIZE,
            unit="eV/atom",
            weight=1.0,
            required=True,
            rationale=(
                "Use only within one reduced composition and preserve the "
                "MatterSim and CHGNet values separately."
            ),
        ),
        PropertyObjective(
            property_name="max_force",
            direction=ObjectiveDirection.MINIMIZE,
            unit="eV/angstrom",
            weight=0.5,
            required=True,
            rationale="Reject unrelaxed or structurally collapsing candidates.",
        ),
    ],
    constraints=[
        GoalConstraint(
            constraint_id="immutable-chemical-system",
            description=(
                "Every generated crystal must contain exactly the user-requested "
                "element set."
            ),
            property_name="chemical_system",
            operator="eq",
            value=chemical_system,
            hard=True,
        ),
    ],
    validation_profile_id=selected_validation_profile.profile_id,
    candidate_types=[CandidateType.CRYSTAL],
    assumptions=[
        "MatterGen targets are generation conditions, not thermodynamic validation.",
        "Cross-stoichiometry absolute MLIP energies are not comparable.",
        "The local Fusion backend is a deterministic evidence controller, not a learned predictor.",
        GENERIC_T4_FIELD_PROPERTY_BOUNDARY,
    ],
    max_cycles=SEARCH_ROUNDS,
)
base_controls = GenerationControls(
    alpha=BASE_GUIDANCE_ALPHA,
    temperature=1.0,
    mutation_strength=0.0,
    diversity_strength=0.3,
    schedule_step=0,
    decision_reason=(
        "MatterGen conditioned baseline: alpha 0.5 maps to classifier-free-guidance gamma 2"
    ),
)
search_config = WorkspaceRunConfig(
    workspace_mode=WorkspaceMode.ON,
    seed=BASE_SEED,
    generator_seed=BASE_SEED,
    goal_hash=stable_hash(search_goal),
    parent_candidate_ref=parent.candidate_ref,
    pair_key="colab-material-adaptive-search",
    cohort_index=0,
    generator_id="mattergen",
    generator_version=mattergen_component.install.version,
    generator_code_revision=mattergen_component.source.revision,
    generator_weight_revision=mattergen_weight_revision,
    generator_parameters_hash=stable_hash({
        "pretrained_name": os.environ["MATTERGEN_PRETRAINED_NAME"],
        "chemical_system": chemical_system,
        "base_generation_conditions": base_conditions,
        "global_search_plan": SEARCH_PLAN,
    }),
    decoder_config_hash=stable_hash({"decoder": "mattergen-upstream-cif"}),
    postprocessing_hash=stable_hash({
        "authoritative_representation": "candidate-cif-only",
        "cross_round_structure_match": "deferred-to-notebook",
    }),
    resource_budget_hash=stable_hash({
        "runtime": "colab-t4",
        "max_generation_calls": MAX_GENERATION_CALLS,
        "max_generated_candidates": TOTAL_CANDIDATES,
    }),
    evaluator_panel_hash=stable_hash({"experts": ["mattersim", "chgnet"]}),
    candidate_count=GENERATION_BATCH_SIZE,
    generation_controls=base_controls,
)
GOAL_PATH = RUN_ROOT / "goal.json"
CONFIG_PATH = RUN_ROOT / "run-config.json"
GOAL_PATH.write_text(search_goal.model_dump_json(indent=2), encoding="utf-8")
CONFIG_PATH.write_text(search_config.model_dump_json(indent=2), encoding="utf-8")

stage_evidence_router = build_validation_evidence_router_from_environment(
    artifact_root=RUN_ROOT,
    enabled=ENABLE_STAGE_EVIDENCE,
)
generation_evidence_run = run_stage_evidence_checkpoint(
    request=ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.GENERATION_PRIOR,
        chemical_system=chemical_system,
        material_field=domain_plan.profile.material_field,
        application_subtype=domain_plan.resolution.application_subtype,
        problem_context=material_problem_context,
        observations={
            "user_prompt": DISCOVERY_PROMPT.strip(),
            "base_generation_conditions": base_conditions.model_dump(mode="json"),
            "search_plan": SEARCH_PLAN,
            "requested_candidate_count": TOTAL_CANDIDATES,
            "generation_model": "MatterGen",
            "numeric_property_scores_created": False,
        },
        focus=(
            "Retrieve experimentally reported phases, failed or negative synthesis "
            "evidence, stability constraints, and supported MatterGen conditions."
        ),
    ),
    goal=search_goal,
    runtime_validator_state={
        "authority_outcomes": {
            "mattergen-supported-condition-allowlist": "completed",
            "evidence-driven-fusion-controller": "pending",
        },
        "summary": {
            "adaptive_search": True,
            "requested_rounds": SEARCH_ROUNDS,
            "requested_candidate_count": TOTAL_CANDIDATES,
            "base_guidance_alpha": BASE_GUIDANCE_ALPHA,
            "base_classifier_free_guidance_gamma": BASE_CFG_GAMMA,
            "supported_generation_fields": [
                "chemical_system", "energy_above_hull", "guidance_alpha"
            ],
            "generation_executed": False,
        },
        "unknown_is_pass": False,
    },
    validator_output_artifacts=[PARENT_PATH, GOAL_PATH, CONFIG_PATH],
)

generation_rag_bundle_path = None
if generation_evidence_run.report.handoff.can_steer_generation:
    if generation_evidence_run.bundle is None:
        raise AssertionError("Generation handoff permits steering but has no bundle.")
    if not generation_evidence_run.report.bundle_relative_path:
        raise AssertionError("Generation evidence bundle path is missing.")
    generation_rag_bundle_path = (
        RUN_ROOT / generation_evidence_run.report.bundle_relative_path
    ).resolve()
    generation_rag_bundle_path.relative_to(RUN_ROOT.resolve())
    if not generation_rag_bundle_path.is_file():
        raise AssertionError("Persisted generation evidence bundle is missing.")

print(json.dumps({
    "goal": search_goal.goal_id,
    "search_plan": SEARCH_PLAN,
    "base_conditions": base_conditions.model_dump(mode="json"),
    "generation_rag_bundle": (
        str(generation_rag_bundle_path.relative_to(RUN_ROOT))
        if generation_rag_bundle_path is not None else None
    ),
}, indent=2))


In [ ]:
# @title 3. Run the real global-budgeted multi-round Fusion search
from discovery_os.artifacts import ArtifactStore
from discovery_os.fusion_exploration import ExpertEvidenceStore
from discovery_os.fusion_search import PersistedFusionSearchReport

requested_count = TOTAL_CANDIDATES
CREDENTIAL_ENV_MARKERS = ("KEY", "TOKEN", "SECRET", "PASSWORD", "CREDENTIAL")
credential_env_values = {
    name: value
    for name, value in os.environ.items()
    if value and any(marker in name.upper() for marker in CREDENTIAL_ENV_MARKERS)
}
cli_environment = {
    name: value
    for name, value in os.environ.items()
    if name not in credential_env_values
}

command = [
    sys.executable, "-m", "discovery_os.cli", "fusion-search",
    "--search-id", RUN_ID + "-adaptive-search",
    "--goal", str(GOAL_PATH),
    "--parent", str(PARENT_PATH),
    "--run-config", str(CONFIG_PATH),
    "--generator", "mattergen",
    "--rounds", str(SEARCH_ROUNDS),
    "--frontier-width", str(FRONTIER_WIDTH),
    "--no-control-sweep",
    "--ranking-limit", str(TOTAL_CANDIDATES),
    "--max-generation-calls", str(MAX_GENERATION_CALLS),
    "--max-generated-candidates", str(TOTAL_CANDIDATES),
    "--expert", "mattersim",
    "--expert", "chgnet",
    "--required-evaluator", "mattersim",
    "--required-evaluator", "chgnet",
    "--artifacts", str(ARTIFACT_ROOT),
    "--workspace-id", "colab-material-adaptive-search",
]
if generation_rag_bundle_path is not None:
    command.extend(["--rag-bundle", str(generation_rag_bundle_path)])

print("$", " ".join(shlex.quote(part) for part in command))
completed = subprocess.run(
    command,
    cwd=REPOSITORY_DIR,
    env=cli_environment,
    check=True,
    text=True,
    capture_output=True,
)
persisted_search = PersistedFusionSearchReport.model_validate_json(
    completed.stdout,
    strict=True,
)
search_report = persisted_search.report
FUSION_SEARCH_REPORT_PATH = RUN_ROOT / "fusion-search.json"
FUSION_SEARCH_REPORT_PATH.write_text(
    persisted_search.model_dump_json(indent=2) + "\n",
    encoding="utf-8",
)

generated_search_records = sorted(
    (
        record
        for record in search_report.candidate_records
        if record.generation_provenance is not None
    ),
    key=lambda record: (
        record.round_index,
        str(record.source_branch),
        record.record_id,
    ),
)
returned_candidate_count = len(generated_search_records)
if returned_candidate_count != search_report.budget_usage.generated_candidates:
    raise RuntimeError("Search report candidate records disagree with budget usage.")
if returned_candidate_count == 0:
    raise RuntimeError("The multi-round search generated no candidates.")
if returned_candidate_count > TOTAL_CANDIDATES:
    raise RuntimeError("The global generated-candidate budget was exceeded.")

evidence_store = ExpertEvidenceStore(ArtifactStore(ARTIFACT_ROOT))
generated_rows = []
search_generation_funnels = {}
search_generation_funnel_hashes = {}
raw_exact_file_hashes = set()
raw_model_structure_count = 0
parsed_model_structure_count = 0

for generated_index, record in enumerate(generated_search_records):
    candidate = record.candidate
    feature_refs = []
    for evidence_id in record.evidence_ids:
        envelope = evidence_store.load(evidence_id)
        if envelope.feature_ref.candidate_ref != candidate.candidate_ref:
            raise RuntimeError("Evidence Store candidate binding changed.")
        if envelope.feature_ref.expert_id in {"mattersim", "chgnet"}:
            feature_refs.append(envelope.feature_ref)
    if {reference.expert_id for reference in feature_refs} != {
        "mattersim", "chgnet"
    }:
        raise RuntimeError("Every generated candidate requires MatterSim and CHGNet evidence.")

    funnel = candidate.attributes.get("generation_funnel")
    funnel_hashes = candidate.attributes.get("generation_funnel_hashes")
    if not isinstance(funnel, dict) or not isinstance(funnel_hashes, dict):
        raise RuntimeError("MatterGen generation funnel provenance is missing.")
    required_funnel_keys = {
        "requested_samples", "raw_model_structures", "parsed_structures",
        "exact_file_unique", "crystallographically_unique", "geometry_valid",
    }
    if not required_funnel_keys.issubset(funnel):
        raise RuntimeError("MatterGen generation funnel is incomplete.")
    if int(funnel["requested_samples"]) != GENERATION_BATCH_SIZE:
        raise RuntimeError("Per-call MatterGen batch size differs from the run config.")
    if int(funnel["crystallographically_unique"]) != GENERATION_BATCH_SIZE:
        raise RuntimeError("MatterGen did not return the requested per-call unique batch.")

    exact_hashes = funnel_hashes.get("exact_file_sha256s")
    if (
        not isinstance(exact_hashes, list)
        or len(set(exact_hashes)) != int(funnel["exact_file_unique"])
        or any(
            not isinstance(value, str)
            or re.fullmatch(r"[0-9a-f]{64}", value) is None
            for value in exact_hashes
        )
    ):
        raise RuntimeError("MatterGen exact-file hash audit is invalid.")

    search_generation_funnels[record.record_id] = dict(funnel)
    search_generation_funnel_hashes[record.record_id] = dict(funnel_hashes)
    raw_exact_file_hashes.update(exact_hashes)
    raw_model_structure_count += int(funnel["raw_model_structures"])
    parsed_model_structure_count += int(funnel["parsed_structures"])

    branch_id = str(record.source_branch)
    step_id = f"round-{record.round_index:02d}-{branch_id}"
    reported_conditions = candidate.attributes.get("conditions")
    if not isinstance(reported_conditions, dict):
        reported_conditions = {}
    target_hull = reported_conditions.get("energy_above_hull")
    conditions = GenerationConditions(
        profile_id=step_id,
        guidance_alpha=record.generation_controls.alpha,
        target_energy_above_hull_eV_atom=(
            float(target_hull) if target_hull is not None else None
        ),
    )
    generated_rows.append({
        "profile_id": step_id,
        "search_record_id": record.record_id,
        "round_index": record.round_index,
        "source_branch": branch_id,
        "conditions": conditions,
        "candidate": candidate,
        "feature_refs": feature_refs,
        "generation_provenance": record.generation_provenance.model_dump(mode="json"),
        "generation_controls": record.generation_controls.model_dump(mode="json"),
        "pair_slot": {
            "pair_slot": generated_index,
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "batch_seed": record.generation_provenance.seed,
            "stream_position": 0,
        },
    })

round_zero_rows = [row for row in generated_rows if row["round_index"] == 0]
if not round_zero_rows:
    raise RuntimeError("The adaptive search is missing its baseline round.")
for row in round_zero_rows:
    if row["generation_controls"]["alpha"] != BASE_GUIDANCE_ALPHA:
        raise RuntimeError("The baseline MatterGen alpha changed.")
    applied = row["candidate"].attributes.get("applied_generation_controls", {})
    if applied.get("diffusion_guidance_factor") != BASE_CFG_GAMMA:
        raise RuntimeError("MatterGen alpha 0.5 must map to applied CFG gamma 2.")

applied_generation_profiles = []
for row in generated_rows:
    requested_alpha = float(row["generation_controls"]["alpha"])
    applied = row["candidate"].attributes.get("applied_generation_controls")
    if not isinstance(applied, dict):
        raise RuntimeError("MatterGen applied-generation-control provenance is missing.")
    applied_conditions = applied.get("conditions")
    if not isinstance(applied_conditions, dict) or not applied_conditions:
        raise RuntimeError("Every material-search candidate requires applied conditions.")
    gamma = applied.get("diffusion_guidance_factor")
    alpha_to_gamma_scale = applied.get("alpha_to_gamma_scale")
    if (
        isinstance(gamma, bool)
        or not isinstance(gamma, (int, float))
        or isinstance(alpha_to_gamma_scale, bool)
        or not isinstance(alpha_to_gamma_scale, (int, float))
    ):
        raise RuntimeError("MatterGen applied alpha/gamma provenance is invalid.")
    expected_gamma = round(requested_alpha * float(alpha_to_gamma_scale), 8)
    if abs(float(gamma) - expected_gamma) > 1.0e-8:
        raise RuntimeError("MatterGen applied gamma differs from alpha times its scale.")
    applied_target = applied_conditions.get("energy_above_hull")
    if applied_target is not None and (
        isinstance(applied_target, bool)
        or not isinstance(applied_target, (int, float))
    ):
        raise RuntimeError("Applied energy-above-hull target must be numeric or null.")
    applied_generation_profiles.append({
        "search_record_id": row["search_record_id"],
        "round_index": row["round_index"],
        "source_branch": row["source_branch"],
        "requested_alpha": requested_alpha,
        "applied_cfg_gamma": float(gamma),
        "alpha_to_gamma_scale": float(alpha_to_gamma_scale),
        "target_energy_above_hull_eV_atom": (
            float(applied_target) if applied_target is not None else None
        ),
        "applied_conditions": dict(applied_conditions),
        "requested_temperature": row["generation_controls"]["temperature"],
        "requested_mutation_strength": row["generation_controls"][
            "mutation_strength"
        ],
        "requested_diversity_strength": row["generation_controls"][
            "diversity_strength"
        ],
    })

applied_alphas = {item["requested_alpha"] for item in applied_generation_profiles}
applied_gammas = {item["applied_cfg_gamma"] for item in applied_generation_profiles}
applied_target_profiles = {
    item["target_energy_above_hull_eV_atom"]
    for item in applied_generation_profiles
    if item["target_energy_above_hull_eV_atom"] is not None
}
if len(applied_alphas) < 2 or len(applied_gammas) < 2:
    raise RuntimeError("The real search did not apply multiple alpha/gamma profiles.")
if len(applied_target_profiles) < 3:
    raise RuntimeError("The real search did not apply at least three hull-target profiles.")
GENERATION_PROFILE_MATRIX_PATH = RUN_ROOT / "generation-profile-matrix.json"
GENERATION_PROFILE_MATRIX_PATH.write_text(
    json.dumps({
        "mapping": "gamma=alpha*alpha_to_gamma_scale when conditioned",
        "unique_applied_alphas": sorted(applied_alphas),
        "unique_applied_cfg_gammas": sorted(applied_gammas),
        "unique_target_energy_above_hull_eV_atom": sorted(applied_target_profiles),
        "profiles": applied_generation_profiles,
    }, indent=2) + "\n",
    encoding="utf-8",
)

source_closed_evidence_cycles = sum(
    cycle.evidence_branch_id is not None for cycle in search_report.cycle_records
)
if generation_rag_bundle_path is not None and source_closed_evidence_cycles == 0:
    raise RuntimeError("A usable generation RAG bundle was not bound to a search cycle.")

finalize_stage_runtime_validator(
    ValidationEvidenceStage.GENERATION_PRIOR,
    runtime_validator_state={
        "authority_outcomes": {
            "mattergen-supported-condition-allowlist": "completed",
            "evidence-driven-fusion-controller": "completed",
        },
        "summary": {
            "fusion_search_calls": search_report.budget_usage.generation_calls,
            "rounds_requested": search_report.rounds_requested,
            "rounds_completed": search_report.rounds_completed,
            "generated_candidate_count": returned_candidate_count,
            "global_candidate_budget": TOTAL_CANDIDATES,
            "source_closed_evidence_cycles": source_closed_evidence_cycles,
            "generation_rag_bundle_used": generation_rag_bundle_path is not None,
            "adaptive_scheduler_histories": {
                str(branch.branch): len(branch.scheduler_history)
                for branch in search_report.branches
            },
        },
        "unknown_is_pass": False,
    },
    validator_output_artifacts=[
        FUSION_SEARCH_REPORT_PATH,
        GENERATION_PROFILE_MATRIX_PATH,
        ARTIFACT_ROOT / persisted_search.report_artifact.relative_path,
    ],
)
print(json.dumps({
    "search_status": str(search_report.status),
    "rounds_completed": search_report.rounds_completed,
    "generation_calls": search_report.budget_usage.generation_calls,
    "requested_samples": requested_count,
    "returned_candidates": returned_candidate_count,
    "raw_model_structures": raw_model_structure_count,
    "parsed_structures": parsed_model_structure_count,
    "global_raw_exact_file_unique": len(raw_exact_file_hashes),
    "unique_applied_alphas": sorted(applied_alphas),
    "unique_applied_cfg_gammas": sorted(applied_gammas),
    "unique_hull_target_profiles": sorted(applied_target_profiles),
    "branches": [str(branch.branch) for branch in search_report.branches],
}, indent=2))


In [ ]:
# @title 4. Apply the exact-file and cross-round crystal-identity funnel
from dataclasses import asdict

from discovery_os.crystal_identity import (
    canonicalize_crystal_structure,
    exact_file_hash,
    group_crystal_structures,
    parse_crystal_structure,
    validate_crystal_geometry,
)

from discovery_os.novelty import (
    MaterialsProjectStructureLookup,
    StagedNoveltyAssessor,
)

identity_failures = []
parsed_rows = []
for row in generated_rows:
    candidate = row["candidate"]
    cif_representations = [
        representation
        for representation in candidate.representations
        if representation.kind == RepresentationKind.CIF
    ]
    if len(cif_representations) != 1:
        identity_failures.append({
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "stage": "parsed",
            "error": "candidate_must_have_exactly_one_authoritative_cif",
        })
        continue
    cif = cif_representations[0]
    try:
        structure = parse_crystal_structure(cif.value, fmt="cif")
    except Exception as exc:
        identity_failures.append({
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "stage": "parsed",
            "error": type(exc).__name__ + ": " + str(exc),
        })
        continue
    digest = exact_file_hash(cif.value)
    output_name = (
        row["profile_id"] + "-" + str(row["pair_slot"]["pair_slot"]).zfill(3)
        + "-" + digest[:12] + ".cif"
    )
    output_path = AUDIT_CIF_ROOT / output_name
    output_path.write_text(cif.value, encoding="utf-8")
    parsed_rows.append({
        **row,
        "cif": cif,
        "structure": structure,
        "exact_file_sha256": digest,
        "raw_audit_cif_path": str(output_path.relative_to(RUN_ROOT)),
    })

exact_unique_rows = []
seen_exact_hashes = set()
for row in parsed_rows:
    if row["exact_file_sha256"] in seen_exact_hashes:
        continue
    seen_exact_hashes.add(row["exact_file_sha256"])
    exact_unique_rows.append(row)

canonicalizable_rows = []
for row in exact_unique_rows:
    try:
        canonical = canonicalize_crystal_structure(
            row["structure"],
            minimum_distance_angstrom=1.0e-4,
        )
    except Exception as exc:
        identity_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "crystallographically-unique",
            "error": type(exc).__name__ + ": " + str(exc),
        })
        continue
    canonicalizable_rows.append({**row, "canonical": canonical})

grouping = group_crystal_structures(
    tuple(row["canonical"] for row in canonicalizable_rows),
)
crystal_matcher_settings = asdict(grouping.matcher_settings)
crystal_ambiguous_comparisons = [
    {
        **asdict(comparison),
        "left_candidate_ref": canonicalizable_rows[
            comparison.left_index
        ]["candidate"].candidate_ref.model_dump(mode="json"),
        "right_candidate_ref": canonicalizable_rows[
            comparison.right_index
        ]["candidate"].candidate_ref.model_dump(mode="json"),
        "deduplicated": False,
    }
    for comparison in grouping.ambiguous_comparisons
]
crystal_group_records = []
crystal_unique_rows = []
for group_index, group in enumerate(grouping.groups, 1):
    members = [canonicalizable_rows[index] for index in group.member_indices]
    representative = canonicalizable_rows[group.representative_index]
    crystal_unique_rows.append({
        **representative,
        "canonical": grouping.canonical_structures[group.representative_index],
        "crystal_group_id": "crystal-group-" + str(group_index).zfill(4),
    })
    crystal_group_records.append({
        "group_id": "crystal-group-" + str(group_index).zfill(4),
        "representative_ref": representative["candidate"].candidate_ref.model_dump(mode="json"),
        "member_refs": [
            item["candidate"].candidate_ref.model_dump(mode="json") for item in members
        ],
        "member_search_steps": sorted({item["profile_id"] for item in members}),
        "structure_hash": group.representative_hash,
    })

for row in crystal_unique_rows:
    unique_name = (
        row["crystal_group_id"] + "-"
        + row["candidate"].candidate_ref.content_hash[:12] + ".cif"
    )
    unique_path = UNIQUE_CIF_ROOT / unique_name
    unique_path.write_text(row["cif"].value, encoding="utf-8")
    row["unique_cif_path"] = str(unique_path.relative_to(RUN_ROOT))

geometry_valid_rows = []
for row in crystal_unique_rows:
    report = validate_crystal_geometry(
        row["structure"],
        minimum_distance_angstrom=MINIMUM_DISTANCE_A,
        raise_on_error=False,
    )
    if not report.is_valid:
        identity_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "geometry-valid",
            "errors": list(report.errors),
        })
        continue
    geometry_valid_rows.append({
        **row,
        "geometry": {
            "atom_count": report.atom_count,
            "volume_angstrom3": report.volume_angstrom3,
            "volume_per_atom_angstrom3": report.volume_per_atom_angstrom3,
            "minimum_distance_angstrom": report.minimum_distance_angstrom,
            "warnings": list(report.warnings),
        },
        "composition_key": str(
            row["canonical"].canonical_structure.composition.reduced_formula
        ),
    })

identity_novelty_assessments = StagedNoveltyAssessor(
    MaterialsProjectStructureLookup.from_environment()
).assess([row["candidate"] for row in geometry_valid_rows])
identity_novelty_by_ref = {
    ref_key(item.candidate_ref): item for item in identity_novelty_assessments
}
external_novelty_status_counts = {status: 0 for status in ("match", "no_match", "unknown")}
for assessment in identity_novelty_assessments:
    external_novelty_status_counts[str(assessment.external_database.status)] += 1

IDENTITY_CHECKPOINT_PATH = RUN_ROOT / "identity-novelty-validator-output.json"
IDENTITY_CHECKPOINT_PATH.write_text(
    json.dumps({
        "crystal_groups": crystal_group_records,
        "matcher_settings": crystal_matcher_settings,
        "ambiguous_comparisons": crystal_ambiguous_comparisons,
        "geometry_valid_candidate_refs": [
            row["candidate"].candidate_ref.model_dump(mode="json")
            for row in geometry_valid_rows
        ],
        "novelty_assessments": [
            item.model_dump(mode="json") for item in identity_novelty_assessments
        ],
        "unknown_is_pass": False,
    }, indent=2) + "\n",
    encoding="utf-8",
)
identity_evidence_run = run_stage_evidence_checkpoint(
    request=ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.IDENTITY_NOVELTY,
        chemical_system=chemical_system,
        material_field=domain_plan.profile.material_field,
        application_subtype=domain_plan.resolution.application_subtype,
        problem_context=material_problem_context,
        candidate_refs=[row["candidate"].candidate_ref for row in geometry_valid_rows],
        composition_keys=sorted({row["composition_key"] for row in geometry_valid_rows}),
        observations={
            "exact_file_unique": len(exact_unique_rows),
            "crystallographically_unique": len(crystal_unique_rows),
            "geometry_valid": len(geometry_valid_rows),
            "cross_round_structure_groups": len(crystal_group_records),
            "materials_project_external_statuses": external_novelty_status_counts,
            "identity_method": "pymatgen.StructureMatcher strict scale=False",
            "matcher_settings": crystal_matcher_settings,
            "ambiguous_comparison_count": len(crystal_ambiguous_comparisons),
            "absence_is_universal_novelty": False,
        },
        focus=(
            "Retrieve crystallographic reports for the candidate compositions and "
            "possible aliases; database absence must remain scoped or unknown."
        ),
    ),
    goal=search_goal,
    runtime_validator_state={
        "authority_outcomes": {
            "pymatgen-structure-matcher": "completed",
            "materials-project-find-structure": (
                "completed"
                if any(
                    external_novelty_status_counts[status] > 0
                    for status in ("match", "no_match")
                )
                else "unknown"
            ),
        },
        "summary": {
            "crystallographic_group_count": len(crystal_group_records),
            "geometry_valid_count": len(geometry_valid_rows),
            "external_structure_statuses": external_novelty_status_counts,
            "absence_interpreted_as_universal_novelty": False,
        },
        "unknown_is_pass": False,
    },
    validator_output_artifacts=[
        IDENTITY_CHECKPOINT_PATH,
        *[
            RUN_ROOT / row["unique_cif_path"]
            for row in geometry_valid_rows
        ],
    ],
)

print({
    "returned_candidate_cifs_parsed": len(parsed_rows),
    "returned_candidate_exact_unique": len(exact_unique_rows),
    "cross_round_crystallographically_unique": len(crystal_unique_rows),
    "geometry_valid": len(geometry_valid_rows),
    "cross_round_groups": len(crystal_group_records),
    "rankable_source_directory": str(UNIQUE_CIF_ROOT),
    "raw_audit_directory_not_ranked": str(AUDIT_CIF_ROOT),
})


In [ ]:
# @title 5. Preserve raw expert evidence, relax twice, and rank by composition
import math

from discovery_os.fusion_schemas import ExpertFeaturePayload, FeatureStatus
from discovery_os.mlip_reliability import (
    CompositionEnergyPair,
    CompositionRelativeEnergyDisagreement,
    composition_relative_energy_disagreement,
)
from discovery_os.hashing import bytes_hash
from discovery_os.materials_screening import (
    CandidateScreeningVector,
    MLIPScreeningPrediction,
    classify_model_disagreement,
    force_rmse,
    rank_composition_scoped_pareto,
    select_dft_handoff_refs,
)
from discovery_os.sidecars.conversions import periodic_atom_entity_ids
from discovery_os.relaxation import (
    PeriodicRelaxationPayload,
    PeriodicRelaxationRequest,
    PeriodicRelaxationSettings,
)

def properties_by_name(payload: ExpertFeaturePayload) -> dict:
    return {item.property_name: item for item in payload.properties}

def required_prediction(payload: ExpertFeaturePayload) -> MLIPScreeningPrediction:
    properties = properties_by_name(payload)
    energy = properties.get("energy_per_atom")
    maximum_force = properties.get("max_force")
    if energy is None or maximum_force is None:
        raise ValueError(payload.expert_id + " omitted energy_per_atom or max_force.")
    if energy.unit != "eV/atom" or maximum_force.unit != "eV/angstrom":
        raise ValueError(payload.expert_id + " returned unexpected property units.")
    stress = properties.get("stress_norm")
    return MLIPScreeningPrediction(
        expert_id=payload.expert_id,
        energy_per_atom_eV=energy.value,
        max_force_eV_A=maximum_force.value,
        stress_norm=(stress.value if stress is not None else None),
        stress_unit=(stress.unit if stress is not None else None),
    )

def load_verified_feature(reference):
    path = ARTIFACT_ROOT / reference.artifact.relative_path
    resolved = path.resolve()
    resolved.relative_to(ARTIFACT_ROOT.resolve())
    encoded = resolved.read_bytes()
    if len(encoded) != reference.artifact.byte_size:
        raise ValueError("Feature artifact byte size changed.")
    if bytes_hash(encoded) != reference.artifact.sha256:
        raise ValueError("Feature artifact SHA-256 changed.")
    payload = ExpertFeaturePayload.model_validate_json(encoded, strict=True)
    if payload.candidate_ref != reference.candidate_ref:
        raise ValueError("Feature artifact candidate binding changed.")
    if payload.expert_id != reference.expert_id:
        raise ValueError("Feature artifact expert binding changed.")
    return payload

def aligned_force_rows(
    first: ExpertFeaturePayload,
    second: ExpertFeaturePayload,
    candidate: Candidate,
):
    if first.tensor is None or second.tensor is None:
        raise ValueError("Successful MLIP evidence requires force tensors.")
    if first.semantics is None or second.semantics is None:
        raise ValueError("Successful MLIP evidence requires semantics.")
    first_ids = list(first.semantics.entity_ids)
    second_ids = list(second.semantics.entity_ids)
    cif_rows = [
        item for item in candidate.representations
        if item.kind == RepresentationKind.CIF
    ]
    if len(cif_rows) != 1:
        raise ValueError("Force alignment requires one authoritative Candidate CIF.")
    source_structure = parse_crystal_structure(cif_rows[0].value, fmt="cif")
    expected_ids = list(periodic_atom_entity_ids(source_structure))
    if (
        not first_ids
        or len(first_ids) != len(set(first_ids))
        or set(first_ids) != set(second_ids)
        or set(first_ids) != set(expected_ids)
    ):
        raise ValueError(
            "MLIP site identities differ from the authoritative CIF species/coordinates."
        )
    ordered_site_signature = [
        {
            "entity_id": entity_id,
            "species": str(site.species),
            "fractional_coordinates": [
                round(float(value), 12) for value in site.frac_coords
            ],
        }
        for entity_id, site in zip(expected_ids, source_structure, strict=True)
    ]
    def rows(payload):
        columns = payload.tensor.shape[-1]
        if len(payload.tensor.shape) != 2 or columns < 3:
            raise ValueError("Expected an atom-by-feature MLIP tensor.")
        values = payload.tensor.values
        return [
            values[index:index + columns]
            for index in range(0, len(values), columns)
        ]
    first_rows = rows(first)
    second_rows = rows(second)
    second_by_id = dict(zip(second_ids, second_rows, strict=True))
    return (
        first_rows,
        [second_by_id[item] for item in first_ids],
        {
            "basis": "immutable_candidate_ref_plus_species_wrapped_fractional_coordinates",
            "candidate_ref": candidate.candidate_ref.model_dump(mode="json"),
            "ordered_sites": ordered_site_signature,
        },
    )

mlip_rows = []
screening_failures = []
for row in geometry_valid_rows:
    references = {
        reference.expert_id: reference
        for reference in row["feature_refs"]
        if reference.candidate_ref == row["candidate"].candidate_ref
        and reference.expert_id in {"mattersim", "chgnet"}
    }
    if set(references) != {"mattersim", "chgnet"}:
        screening_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "mlip-evaluated",
            "error": "both expert references are required",
        })
        continue
    try:
        payloads = {
            expert_id: load_verified_feature(reference)
            for expert_id, reference in references.items()
        }
        if any(
            payload.status != FeatureStatus.SUCCESS
            for payload in payloads.values()
        ):
            raise ValueError("Both MLIP feature calls must have success status.")
        mattersim_prediction = required_prediction(payloads["mattersim"])
        chgnet_prediction = required_prediction(payloads["chgnet"])
        mattersim_forces, chgnet_forces, force_alignment_audit = aligned_force_rows(
            payloads["mattersim"],
            payloads["chgnet"],
            row["candidate"],
        )
        rmse = force_rmse(mattersim_forces, chgnet_forces)
    except Exception as exc:
        screening_failures.append({
            "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
            "stage": "mlip-evaluated",
            "error": type(exc).__name__ + ": " + str(exc),
        })
        continue
    mlip_rows.append({
        **row,
        "expert_payloads": payloads,
        "predictions": {
            "mattersim": mattersim_prediction,
            "chgnet": chgnet_prediction,
        },
        "force_rmse_eV_A": rmse,
        "force_alignment_audit": force_alignment_audit,
        "feature_artifacts": {
            expert_id: references[expert_id].artifact.model_dump(mode="json")
            for expert_id in ("mattersim", "chgnet")
        },
    })

relaxation_settings = PeriodicRelaxationSettings(
    optimizer="FIRE",
    requested_steps=RELAX_STEPS,
    target_fmax_eV_A=RELAX_FMAX_EV_A,
    relax_cell=True,
    minimum_distance_safety_A=MINIMUM_DISTANCE_A,
    max_abs_volume_change_fraction=0.35,
)
relaxation_rows = []
for row_index, row in enumerate(mlip_rows):
    records = {}
    payloads = {}
    for expert_id, variable in (
        ("mattersim", "MATTERSIM_API_URL"),
        ("chgnet", "CHGNET_API_URL"),
    ):
        request = PeriodicRelaxationRequest(
            candidate=row["candidate"],
            settings=relaxation_settings,
            seed=BASE_SEED + row_index,
        )
        endpoint = os.environ[variable].rstrip("/") + "/v1/relax"
        try:
            response = requests.post(
                endpoint,
                json=request.model_dump(mode="json"),
                timeout=1800,
            )
            response.raise_for_status()
            payload = PeriodicRelaxationPayload.model_validate_json(
                response.content,
                strict=True,
            )
            if payload.candidate_ref != row["candidate"].candidate_ref:
                raise ValueError("Relaxation response cites another candidate.")
            if payload.expert_id != expert_id:
                raise ValueError("Relaxation response cites another expert.")
            payloads[expert_id] = payload
            records[expert_id] = {
                "endpoint_operation": "/v1/relax",
                "request_executed": True,
                "execution_succeeded": payload.execution_succeeded,
                "converged": payload.converged,
                "strict_gate_passed": payload.strict_gate_passed,
                "payload": payload.model_dump(mode="json"),
                "error": None,
            }
        except Exception as exc:
            records[expert_id] = {
                "endpoint_operation": "/v1/relax",
                "request_executed": True,
                "execution_succeeded": False,
                "converged": False,
                "strict_gate_passed": False,
                "payload": None,
                "error": type(exc).__name__ + ": " + str(exc),
            }

    both_converged = (
        set(payloads) == {"mattersim", "chgnet"}
        and all(
            payload.converged and payload.strict_gate_passed
            for payload in payloads.values()
        )
    )
    relaxed_structure_match = None
    relaxed_structure_comparison_error = None
    relaxed_structure_identity_audit = None
    relaxed_atom_count = None
    relaxed_predictions = None
    if set(payloads) == {"mattersim", "chgnet"}:
        try:
            relaxed_structures = {
                expert_id: parse_crystal_structure(
                    payload.relaxed_structure.value,
                    fmt="cif",
                )
                for expert_id, payload in payloads.items()
            }
            atom_counts = {
                expert_id: len(structure)
                for expert_id, structure in relaxed_structures.items()
            }
            expected_atom_count = int(row["geometry"]["atom_count"])
            if set(atom_counts.values()) != {expected_atom_count}:
                raise ValueError(
                    "Relaxed atom counts differ from the authoritative candidate CIF."
                )
            relaxed_compositions = {
                expert_id: str(structure.composition.reduced_formula)
                for expert_id, structure in relaxed_structures.items()
            }
            if set(relaxed_compositions.values()) != {row["composition_key"]}:
                raise ValueError(
                    "Relaxation changed composition or expert composition identity."
                )
            relaxed_atom_count = expected_atom_count
            relaxed_grouping = group_crystal_structures(
                tuple(relaxed_structures[expert] for expert in ("mattersim", "chgnet"))
            )
            relaxed_structure_identity_audit = {
                "matcher_settings": asdict(relaxed_grouping.matcher_settings),
                "ambiguous_comparisons": [
                    asdict(item) for item in relaxed_grouping.ambiguous_comparisons
                ],
            }
            relaxed_structure_match = (
                None
                if relaxed_grouping.ambiguous_comparisons
                else len(relaxed_grouping.groups) == 1
            )
            if both_converged:
                relaxed_predictions = {
                    expert_id: MLIPScreeningPrediction(
                        expert_id=expert_id,
                        energy_per_atom_eV=(
                            payload.final_energy_eV / relaxed_atom_count
                        ),
                        max_force_eV_A=payload.final_max_force_eV_A,
                        stress_norm=None,
                        stress_unit=None,
                    )
                    for expert_id, payload in payloads.items()
                }
                if any(
                    not math.isfinite(prediction.energy_per_atom_eV)
                    or not math.isfinite(prediction.max_force_eV_A)
                    for prediction in relaxed_predictions.values()
                ):
                    raise ValueError("Relaxed ranking properties must be finite.")
        except Exception as exc:
            both_converged = False
            relaxed_predictions = None
            relaxed_structure_match = None
            relaxed_structure_comparison_error = (
                type(exc).__name__ + ": " + str(exc)
            )

    relaxation_rows.append({
        **row,
        "relaxation_records": records,
        "relaxation_payloads": payloads,
        "relaxed_atom_count": relaxed_atom_count,
        "relaxed_predictions": relaxed_predictions,
        "relaxed_structure_match": relaxed_structure_match,
        "relaxed_structure_comparison_error": relaxed_structure_comparison_error,
        "relaxed_structure_identity_audit": relaxed_structure_identity_audit,
        "relaxation_converged": both_converged and relaxed_predictions is not None,
    })

# Construct one immutable, per-composition alignment receipt before computing
# offset-invariant energy/rank disagreement. Singleton composition pools remain
# unknown and are escalated by composition_relative_energy_disagreement().
alignment_panels = []
energy_pairs = []
eligible_by_composition = {}
for row in relaxation_rows:
    if row["relaxation_converged"]:
        eligible_by_composition.setdefault(row["composition_key"], []).append(row)
eligible_relaxed_candidate_count = sum(
    len(rows) for rows in eligible_by_composition.values()
)
singleton_composition_candidate_count = sum(
    len(rows)
    for rows in eligible_by_composition.values()
    if len(rows) == 1
)
singleton_composition_fraction = (
    singleton_composition_candidate_count / eligible_relaxed_candidate_count
    if eligible_relaxed_candidate_count else None
)
SINGLETON_COMPOSITION_FRACTION_THRESHOLD = 0.5
if singleton_composition_fraction is None:
    composition_replication_conclusion = "no_eligible_relaxed_candidates"
elif singleton_composition_fraction > SINGLETON_COMPOSITION_FRACTION_THRESHOLD:
    composition_replication_conclusion = "insufficient_within_composition_replication"
else:
    composition_replication_conclusion = "within_composition_replication_available"
composition_replication_audit = {
    "eligible_relaxed_candidate_count": eligible_relaxed_candidate_count,
    "singleton_composition_candidate_count": singleton_composition_candidate_count,
    "singleton_composition_fraction": singleton_composition_fraction,
    "insufficient_replication_threshold": (
        SINGLETON_COMPOSITION_FRACTION_THRESHOLD
    ),
    "conclusion": composition_replication_conclusion,
    "global_priority_scope": (
        "triage_by_force_diversity_uncertainty_not_cross_formula_stability_ranking"
    ),
    "rag_numeric_stability_substitution_performed": False,
}
for composition_key, composition_rows in sorted(eligible_by_composition.items()):
    ordered_rows = sorted(
        composition_rows,
        key=lambda row: ref_key(row["candidate"].candidate_ref),
    )
    panel_payload = {
        "schema": "aligned-relaxed-composition-energy-panel-v1",
        "composition_key": composition_key,
        "unit": "eV/atom",
        "atom_count_policy": (
            "both relaxed CIF atom counts equal the authoritative candidate CIF atom count"
        ),
        "rows": [
            {
                "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
                "verified_atom_count": row["relaxed_atom_count"],
                "mattersim_final_energy_eV": row["relaxation_payloads"][
                    "mattersim"
                ].final_energy_eV,
                "chgnet_final_energy_eV": row["relaxation_payloads"][
                    "chgnet"
                ].final_energy_eV,
                "mattersim_final_energy_per_atom_eV": row["relaxed_predictions"][
                    "mattersim"
                ].energy_per_atom_eV,
                "chgnet_final_energy_per_atom_eV": row["relaxed_predictions"][
                    "chgnet"
                ].energy_per_atom_eV,
            }
            for row in ordered_rows
        ],
    }
    alignment_artifact_id = stable_hash(panel_payload)
    alignment_panels.append({
        **panel_payload,
        "alignment_artifact_id": alignment_artifact_id,
    })
    for row in ordered_rows:
        energy_pairs.append(CompositionEnergyPair(
            candidate_id=row["candidate"].candidate_ref.candidate_id,
            reduced_composition=composition_key,
            first_model_id="mattersim",
            second_model_id="chgnet",
            first_energy_per_atom_eV=row["relaxed_predictions"][
                "mattersim"
            ].energy_per_atom_eV,
            second_energy_per_atom_eV=row["relaxed_predictions"][
                "chgnet"
            ].energy_per_atom_eV,
            alignment_artifact_id=alignment_artifact_id,
        ))

COMPOSITION_ENERGY_ALIGNMENT_PATH = RUN_ROOT / "composition-energy-alignment.json"
COMPOSITION_ENERGY_ALIGNMENT_PATH.write_text(
    json.dumps({
        "schema": "composition-energy-alignment-index-v1",
        "panels": alignment_panels,
        "composition_replication_audit": composition_replication_audit,
        "raw_cross_model_energy_offsets_used_for_risk": False,
    }, indent=2) + "\n",
    encoding="utf-8",
)
relative_energy_results = composition_relative_energy_disagreement(energy_pairs)
relative_energy_by_candidate_id = {
    item.candidate_id: item for item in relative_energy_results
}
composition_pool_sizes = {
    composition_key: sum(
        row["composition_key"] == composition_key for row in relaxation_rows
    )
    for composition_key in {row["composition_key"] for row in relaxation_rows}
}

classified_relaxation_rows = []
for row in relaxation_rows:
    candidate_id = row["candidate"].candidate_ref.candidate_id
    common_geometry_alignment_id = stable_hash({
        "schema": "common-geometry-mlip-alignment-v1",
        "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
        "force_alignment_audit": row["force_alignment_audit"],
        "feature_artifacts": row["feature_artifacts"],
    })
    relative_energy = relative_energy_by_candidate_id.get(candidate_id)
    if relative_energy is None:
        relative_energy = CompositionRelativeEnergyDisagreement(
            candidate_id=candidate_id,
            reduced_composition=row["composition_key"],
            first_model_id="mattersim",
            second_model_id="chgnet",
            pool_size=max(1, composition_pool_sizes[row["composition_key"]]),
            status="unknown",
            dft_escalation=True,
            uncertainty_reasons=[
                "dual_relaxation_not_converged_or_atom_count_unverified"
            ],
        )
    disagreement = classify_model_disagreement(
        row["predictions"]["mattersim"],
        row["predictions"]["chgnet"],
        force_rmse_eV_A=row["force_rmse_eV_A"],
        relative_energy=relative_energy,
        relaxed_structure_match=row["relaxed_structure_match"],
        require_stress_comparison=True,
        require_relaxed_structure_comparison=True,
    )
    classified_relaxation_rows.append({
        **row,
        "relative_energy_disagreement": relative_energy,
        "disagreement": disagreement,
        "initial_common_geometry_force_rmse_eV_A": row["force_rmse_eV_A"],
        "initial_common_geometry_stress_diff_GPa": (
            disagreement.stress_norm_abs_diff_GPa
        ),
        "initial_common_geometry_alignment_id": common_geometry_alignment_id,
        "disagreement_evidence_geometry": {
            "raw_energy_force_stress": "initial_common_candidate_geometry",
            "energy_risk": "composition_relative_independently_relaxed_geometries",
            "structure_relation": "independently_relaxed_geometry_strict_match",
            "relaxed_force_cross_model_comparison_performed": False,
            "relaxed_stress_cross_model_comparison_performed": False,
        },
    })
relaxation_rows = classified_relaxation_rows

vectors = [
    CandidateScreeningVector(
        candidate_ref=row["candidate"].candidate_ref,
        composition_key=row["composition_key"],
        mattersim=row["relaxed_predictions"]["mattersim"],
        chgnet=row["relaxed_predictions"]["chgnet"],
        common_geometry_mattersim=row["predictions"]["mattersim"],
        common_geometry_chgnet=row["predictions"]["chgnet"],
        common_geometry_alignment_id=row["initial_common_geometry_alignment_id"],
        disagreement=row["disagreement"],
        geometry_valid=True,
        relaxation_gate_passed=True,
    )
    for row in relaxation_rows
    if row["relaxation_converged"] and row["relaxed_predictions"] is not None
]
all_pareto_ranks = rank_composition_scoped_pareto(vectors)
vector_by_ref = {ref_key(item.candidate_ref): item for item in vectors}
rank_by_ref = {ref_key(item.candidate_ref): item for item in all_pareto_ranks}
strict_ranked = list(all_pareto_ranks)
relaxation_converged_rows = [
    row for row in relaxation_rows if row["relaxation_converged"]
]

disagreement_risk_counts = {risk: 0 for risk in ("low", "medium", "high")}
relative_energy_status_counts = {status: 0 for status in ("available", "unknown")}
for row in relaxation_rows:
    disagreement_risk_counts[str(row["disagreement"].risk)] += 1
    relative_energy_status_counts[str(row["relative_energy_disagreement"].status)] += 1

MLIP_CHECKPOINT_PATH = RUN_ROOT / "mlip-disagreement-validator-output.json"
MLIP_CHECKPOINT_PATH.write_text(
    json.dumps({
        "candidates": [
            {
                "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
                "composition_key": row["composition_key"],
                "initial_feature_predictions": {
                    expert: row["predictions"][expert].model_dump(mode="json")
                    for expert in ("mattersim", "chgnet")
                },
                "relaxed_ranking_predictions": (
                    {
                        expert: row["relaxed_predictions"][expert].model_dump(mode="json")
                        for expert in ("mattersim", "chgnet")
                    }
                    if row["relaxed_predictions"] is not None else None
                ),
                "composition_relative_energy_disagreement": row[
                    "relative_energy_disagreement"
                ].model_dump(mode="json"),
                "initial_common_geometry_force_rmse_eV_A": row[
                    "initial_common_geometry_force_rmse_eV_A"
                ],
                "initial_common_geometry_stress_diff_GPa": row[
                    "initial_common_geometry_stress_diff_GPa"
                ],
                "initial_common_geometry_alignment_id": row[
                    "initial_common_geometry_alignment_id"
                ],
                "disagreement_evidence_geometry": row[
                    "disagreement_evidence_geometry"
                ],
                "force_alignment_audit": row["force_alignment_audit"],
                "disagreement": row["disagreement"].model_dump(mode="json"),
                "feature_artifacts": row["feature_artifacts"],
            }
            for row in relaxation_rows
        ],
        "risk_counts": disagreement_risk_counts,
        "composition_replication_audit": composition_replication_audit,
        "unknown_is_pass": False,
    }, indent=2) + "\n",
    encoding="utf-8",
)
mlip_evidence_run = run_stage_evidence_checkpoint(
    request=ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.MLIP_DISAGREEMENT,
        chemical_system=chemical_system,
        material_field=domain_plan.profile.material_field,
        application_subtype=domain_plan.resolution.application_subtype,
        problem_context=material_problem_context,
        candidate_refs=[row["candidate"].candidate_ref for row in relaxation_rows],
        composition_keys=sorted({row["composition_key"] for row in relaxation_rows}),
        observations={
            "dual_model_evaluated": len(mlip_rows),
            "models": ["MatterSim", "CHGNet"],
            "energy_unit": "eV/atom",
            "force_unit": "eV/angstrom",
            "maximum_composition_relative_energy_difference_eV_atom": (
                max(
                    item.relative_energy_abs_diff_eV_atom
                    for item in relative_energy_results
                    if item.status == "available"
                )
                if any(item.status == "available" for item in relative_energy_results)
                else None
            ),
            "raw_absolute_energy_offset_role": "audit_only_never_risk",
            "relative_energy_statuses": relative_energy_status_counts,
            "composition_alignment_receipt": str(
                COMPOSITION_ENERGY_ALIGNMENT_PATH.relative_to(RUN_ROOT)
            ),
            "composition_replication_audit": composition_replication_audit,
            "maximum_initial_common_geometry_force_rmse_eV_A": (
                max(row["force_rmse_eV_A"] for row in mlip_rows)
                if mlip_rows else None
            ),
            "initial_feature_stress_outputs_available": sum(
                row["predictions"]["mattersim"].stress_norm is not None
                and row["predictions"]["chgnet"].stress_norm is not None
                for row in mlip_rows
            ),
            "disagreement_risks": disagreement_risk_counts,
            "dft_escalation_count": sum(
                row["disagreement"].dft_escalation for row in relaxation_rows
            ),
            "comparison_unknown_count": relative_energy_status_counts["unknown"],
            "property_scores_created_from_literature": False,
        },
        focus=(
            "Retrieve published applicability limits, magnetic or charge-state "
            "caveats, and calculations that could explain the finalized dual-MLIP disagreement."
        ),
    ),
    goal=search_goal,
    runtime_validator_state={
        "authority_outcomes": {
            "mattersim-sidecar": "completed" if mlip_rows else "unknown",
            "chgnet-sidecar": "completed" if mlip_rows else "unknown",
            "cross-model-unit-normalized-disagreement": (
                "completed" if vectors else "unknown"
            ),
        },
        "summary": {
            "evaluated_candidate_count": len(mlip_rows),
            "failed_candidate_count": len(screening_failures),
            "risk_counts": disagreement_risk_counts,
            "missing_comparisons_escalated": True,
        },
        "unknown_is_pass": False,
    },
    validator_output_artifacts=[
        MLIP_CHECKPOINT_PATH,
        COMPOSITION_ENERGY_ALIGNMENT_PATH,
        *[
            ARTIFACT_ROOT / row["feature_artifacts"][expert]["relative_path"]
            for row in mlip_rows
            for expert in ("mattersim", "chgnet")
        ],
    ],
)

RELAXATION_CHECKPOINT_PATH = RUN_ROOT / "relaxation-validator-output.json"
RELAXATION_CHECKPOINT_PATH.write_text(
    json.dumps({
        "settings": relaxation_settings.model_dump(mode="json"),
        "candidates": [
            {
                "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
                "composition_key": row["composition_key"],
                "mattersim": row["relaxation_records"]["mattersim"],
                "chgnet": row["relaxation_records"]["chgnet"],
                "relaxed_structure_match": row["relaxed_structure_match"],
                "relaxed_structure_comparison_error": row[
                    "relaxed_structure_comparison_error"
                ],
                "relaxed_structure_identity_audit": row[
                    "relaxed_structure_identity_audit"
                ],
                "both_converged_and_strict_gate_passed": row["relaxation_converged"],
                "verified_relaxed_atom_count": row["relaxed_atom_count"],
                "relaxed_ranking_predictions": (
                    {
                        expert: row["relaxed_predictions"][expert].model_dump(mode="json")
                        for expert in ("mattersim", "chgnet")
                    }
                    if row["relaxed_predictions"] is not None else None
                ),
            }
            for row in relaxation_rows
        ],
        "unknown_is_pass": False,
    }, indent=2) + "\n",
    encoding="utf-8",
)

def relaxation_authority_state(expert: str) -> str:
    total = len(relaxation_rows)
    successes = sum(
        row["relaxation_records"][expert]["execution_succeeded"]
        for row in relaxation_rows
    )
    if not total or not successes:
        return "unknown"
    return "completed" if successes == total else "partial"

relaxation_evidence_run = run_stage_evidence_checkpoint(
    request=ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.RELAXATION_VALIDATION,
        chemical_system=chemical_system,
        material_field=domain_plan.profile.material_field,
        application_subtype=domain_plan.resolution.application_subtype,
        problem_context=material_problem_context,
        candidate_refs=[row["candidate"].candidate_ref for row in relaxation_rows],
        composition_keys=sorted({row["composition_key"] for row in relaxation_rows}),
        observations={
            "independent_relaxers": ["MatterSim", "CHGNet"],
            "requested_attempts": 2 * len(mlip_rows),
            "execution_failures": sum(
                not row["relaxation_records"][expert]["execution_succeeded"]
                for row in relaxation_rows
                for expert in ("mattersim", "chgnet")
            ),
            "strict_gate_failures": sum(
                not row["relaxation_records"][expert]["strict_gate_passed"]
                for row in relaxation_rows
                for expert in ("mattersim", "chgnet")
            ),
            "both_relaxations_converged": len(relaxation_converged_rows),
            "relaxed_structure_mismatches": sum(
                row["relaxed_structure_match"] is False for row in relaxation_rows
            ),
            "relaxed_structure_comparison_unknown": sum(
                row["relaxed_structure_match"] is None for row in relaxation_rows
            ),
            "disagreement_risks": disagreement_risk_counts,
        },
        focus=(
            "Retrieve known phase transformations, mechanical or dynamical "
            "instability, pressure/temperature dependence, and phonon evidence."
        ),
    ),
    goal=search_goal,
    runtime_validator_state={
        "authority_outcomes": {
            "ase-periodic-optimizer": (
                "completed"
                if relaxation_rows and len(relaxation_converged_rows) == len(relaxation_rows)
                else ("partial" if relaxation_rows else "unknown")
            ),
            "mattersim-relaxation": relaxation_authority_state("mattersim"),
            "chgnet-relaxation": relaxation_authority_state("chgnet"),
        },
        "summary": {
            "candidate_count": len(relaxation_rows),
            "both_relaxations_converged": len(relaxation_converged_rows),
            "strict_gate_is_separate_from_execution": True,
            "unknown_or_failed_is_pass": False,
        },
        "unknown_is_pass": False,
    },
    validator_output_artifacts=[RELAXATION_CHECKPOINT_PATH],
)

print({
    "mlip-evaluated": len(mlip_rows),
    "relaxation-converged": len(relaxation_converged_rows),
    "ranked": len(strict_ranked),
    "high-disagreement-queued": sum(
        row["disagreement"].dft_escalation for row in relaxation_rows
    ),
})


In [ ]:
# @title 6. Assess novelty, escalate disagreement, and create DFT handoffs
from discovery_os.artifacts import ArtifactStore
from discovery_os.dft_handoff import PortablePeriodicDFTInputBackend
from discovery_os.materials_screening import ThermodynamicValidation
from discovery_os.novelty import reserve_external_no_match_portfolio_slot
if not relaxation_rows:
    raise RuntimeError("No geometry-valid candidate completed dual-MLIP feature evaluation.")

novelty_assessments = identity_novelty_assessments
novelty_by_ref = identity_novelty_by_ref

base_dft_refs = select_dft_handoff_refs(
    all_pareto_ranks,
    vectors,
    top_k=DFT_TOP_K,
)
eligible_dft_refs = [item.candidate_ref for item in all_pareto_ranks]
novelty_portfolio = reserve_external_no_match_portfolio_slot(
    base_candidate_refs=base_dft_refs,
    eligible_candidate_refs=eligible_dft_refs,
    assessments=novelty_assessments,
    top_k=DFT_TOP_K,
    max_novelty_slots=1,
)
selected_dft_refs = novelty_portfolio.selected_candidate_refs
if not selected_dft_refs:
    raise RuntimeError(
        "No DFT handoff candidate passed a relaxation gate or high-disagreement escalation rule."
    )
NOVELTY_PORTFOLIO_PATH = RUN_ROOT / "novelty-dft-portfolio-receipt.json"
NOVELTY_PORTFOLIO_PATH.write_text(
    novelty_portfolio.model_dump_json(indent=2) + "\n",
    encoding="utf-8",
)
selected_dft_keys = {ref_key(item) for item in selected_dft_refs}
row_by_ref = {
    ref_key(row["candidate"].candidate_ref): row for row in relaxation_rows
}
selected_dft_candidates = [
    row_by_ref[ref_key(reference)]["candidate"]
    for reference in selected_dft_refs
]
dft_report = PortablePeriodicDFTInputBackend().prepare_inputs(
    selected_dft_candidates,
    artifact_store=ArtifactStore(ARTIFACT_ROOT),
    top_k=len(selected_dft_candidates),
)
if not 1 <= len(dft_report.packages) <= 5:
    raise RuntimeError("A successful shortlist must create 1-5 DFT input packages.")
dft_handoff_artifact_paths = [
    ARTIFACT_ROOT / artifact.relative_path
    for package in dft_report.packages
    for artifact in [package.manifest_artifact, *package.manifest.input_artifacts]
]
dft_evidence_run = run_stage_evidence_checkpoint(
    request=ValidationEvidenceRequest(
        stage=ValidationEvidenceStage.DFT_HANDOFF,
        chemical_system=chemical_system,
        material_field=domain_plan.profile.material_field,
        application_subtype=domain_plan.resolution.application_subtype,
        problem_context=material_problem_context,
        candidate_refs=selected_dft_refs,
        composition_keys=sorted({
            row_by_ref[ref_key(reference)]["composition_key"]
            for reference in selected_dft_refs
        }),
        observations={
            "shortlist_size": len(selected_dft_refs),
            "top_k_limit": DFT_TOP_K,
            "selection": (
                "composition-scoped Pareto plus high-disagreement escalation, with "
                "at most one completed strict external-database no-match slot"
            ),
            "novelty_portfolio_policy": novelty_portfolio.policy,
            "reserved_external_no_match_ref": (
                novelty_portfolio.reserved_external_no_match_ref.model_dump(mode="json")
                if novelty_portfolio.reserved_external_no_match_ref is not None
                else None
            ),
            "unknown_external_novelty_receives_credit": False,
            "prepared_input_packages": len(dft_report.packages),
            "prepared_backend_id": dft_report.backend_id,
            "dft_executed": dft_report.calculation_executed,
            "pseudopotentials_included": False,
            "required_review": [
                "reference phases",
                "magnetic ordering",
                "functional and Hubbard-U choices",
                "pseudopotentials",
                "cutoff and k-point convergence",
                "phonons",
            ],
        },
        focus=(
            "Retrieve calculation choices and experimental references needed to "
            "review the persisted portable DFT manifests; do not claim a DFT result."
        ),
    ),
    goal=search_goal,
    runtime_validator_state={
        "authority_outcomes": {
            "periodic-dft-backend-contract": "completed",
            "external-pseudopotential-review": "pending",
            "reference-phase-convergence-review": "pending",
        },
        "summary": {
            "prepared_package_count": len(dft_report.packages),
            "calculation_executed": dft_report.calculation_executed,
            "pseudopotentials_included": False,
            "uncalculated_properties_remain_null": True,
        },
        "unknown_is_pass": False,
    },
    validator_output_artifacts=[
        NOVELTY_PORTFOLIO_PATH,
        *dft_handoff_artifact_paths,
    ],
)
stage_evidence_refs = {
    stage: {
        "report_id": run.report.report_id,
        "stage": str(run.report.stage),
        "status": str(run.report.status),
        "material_field": domain_plan.profile.material_field.value,
        "material_domain_profile_id": domain_plan.profile.profile_id,
        "domain_route": run.report.domain_route.model_dump(mode="json"),
        "report_relative_path": str(run.report_path.relative_to(RUN_ROOT)),
        "checkpoint_receipt_relative_path": str(
            stage_checkpoint_receipt_paths[stage].relative_to(RUN_ROOT)
        ),
        "record_count": run.report.record_count,
        "claim_count": run.report.claim_count,
        "branch_count": run.report.branch_count,
        "source_statuses": [
            item.model_dump(mode="json") for item in run.report.source_statuses
        ],
        "official_validators": list(run.report.route.official_validators),
        "validator_authorities": [
            item.model_dump(mode="json")
            for item in run.report.route.validator_authorities
        ],
        "mcp_contract": run.report.route.mcp_contract.model_dump(mode="json"),
        "mcp_contract_status": str(run.report.mcp_contract_status),
        "mcp_tool_name": run.report.mcp_tool_name,
        "handoff": run.report.handoff.model_dump(mode="json"),
        "runtime_validator": stage_checkpoint_receipts[stage]["runtime_validator"],
        "property_score_created": run.report.property_score_created,
    }
    for stage, run in stage_evidence_runs.items()
}

dft_package_by_ref = {
    ref_key(package.manifest.candidate_ref): package
    for package in dft_report.packages
}
thermodynamic_nulls = ThermodynamicValidation().model_dump(mode="json")
screening_records = []
for row in relaxation_rows:
    key = ref_key(row["candidate"].candidate_ref)
    rank = rank_by_ref.get(key)
    novelty = novelty_by_ref[key]
    package = dft_package_by_ref.get(key)
    generator_provenance = row["generation_provenance"]
    screening_records.append({
        "candidate_ref": row["candidate"].candidate_ref.model_dump(mode="json"),
        "composition_key": row["composition_key"],
        "material_field_validation": {
            "material_field": domain_plan.profile.material_field.value,
            "profile_id": domain_plan.profile.profile_id,
            "generic_t4_candidate_type": "crystal",
            "generic_mlip_screening_performed": True,
            "field_specific_property_calculation_executed": False,
            "unexecuted_required_properties": field_property_status,
            "scientific_boundary": GENERIC_T4_FIELD_PROPERTY_BOUNDARY,
        },
        "authoritative_candidate_cif": {
            "relative_path": row["unique_cif_path"],
            "raw_audit_relative_path": row["raw_audit_cif_path"],
            "exact_file_sha256": row["exact_file_sha256"],
            "crystal_group_id": row["crystal_group_id"],
            "source": "Candidate.representations[kind=cif]",
            "final_export_scope": "cross_round_crystallographic_representative",
            "raw_audit_cif_counted_or_ranked": False,
            "zip_or_extxyz_merge_performed": False,
        },
        "generation_conditions": {
            "chemical_system": chemical_system,
            "adaptive_search_step": row["conditions"].model_dump(mode="json"),
            "candidate_reported_conditions": row["candidate"].attributes.get(
                "conditions"
            ),
            "generator_provenance": generator_provenance,
            "scientific_role": "generation_target_not_a_measured_property",
        },
        "screening_predictions": {
            "initial_feature_predictions": {
                expert: row["predictions"][expert].model_dump(mode="json")
                for expert in ("mattersim", "chgnet")
            },
            "relaxed_ranking_predictions": (
                {
                    expert: row["relaxed_predictions"][expert].model_dump(mode="json")
                    for expert in ("mattersim", "chgnet")
                }
                if row["relaxed_predictions"] is not None else None
            ),
            "composition_relative_energy_disagreement": row[
                "relative_energy_disagreement"
            ].model_dump(mode="json"),
            "raw_feature_artifacts": row["feature_artifacts"],
            "initial_common_geometry_force_rmse_eV_A": row[
                "initial_common_geometry_force_rmse_eV_A"
            ],
            "initial_common_geometry_stress_diff_GPa": row[
                "initial_common_geometry_stress_diff_GPa"
            ],
            "initial_common_geometry_alignment_id": row[
                "initial_common_geometry_alignment_id"
            ],
            "disagreement_evidence_geometry": row[
                "disagreement_evidence_geometry"
            ],
            "force_alignment_audit": row["force_alignment_audit"],
            "model_disagreement": row["disagreement"].model_dump(mode="json"),
            "composition_scoped_pareto": (
                rank.model_dump(mode="json") if rank is not None else None
            ),
            "entered_final_ranked_funnel": rank is not None,
            "cross_stoichiometry_energy_comparison_performed": False,
        },
        "relaxation_validation": {
            "mattersim": row["relaxation_records"]["mattersim"],
            "chgnet": row["relaxation_records"]["chgnet"],
            "both_converged_and_strict_gate_passed": row["relaxation_converged"],
            "verified_relaxed_atom_count": row["relaxed_atom_count"],
            "relaxed_structure_match": row["relaxed_structure_match"],
            "relaxed_structure_identity_audit": row[
                "relaxed_structure_identity_audit"
            ],
            "relaxed_structure_comparison_error": row[
                "relaxed_structure_comparison_error"
            ],
        },
        "novelty": novelty.model_dump(mode="json"),
        "stage_validation_evidence": stage_evidence_refs,
        "dft_escalation": {
            "required_due_to_high_disagreement": row[
                "disagreement"
            ].dft_escalation,
            "queue_status": (
                "packaged"
                if key in selected_dft_keys
                else (
                    "unranked_failed_relaxation_requires_operator_review"
                    if not row["relaxation_converged"]
                    else (
                        "queued_by_disagreement_but_outside_top_k"
                        if row["disagreement"].dft_escalation
                        else "not_selected"
                    )
                )
            ),
            "input_package_manifest": (
                package.manifest_artifact.model_dump(mode="json")
                if package is not None else None
            ),
        },
        "thermodynamic_validation": {
            "status": "not_run",
            "dft_executed": False,
            **thermodynamic_nulls,
        },
    })

funnel = {
    "requested_samples": requested_count,
    "raw_model_structures": raw_model_structure_count,
    "parsed_structures": parsed_model_structure_count,
    "exact_file_unique": len(raw_exact_file_hashes),
    "crystallographically_unique": len(crystal_unique_rows),
    "geometry_valid": len(geometry_valid_rows),
    "mlip_evaluated": len(mlip_rows),
    "relaxation_converged": len(relaxation_converged_rows),
    "ranked_candidates": len(strict_ranked),
}
failures = identity_failures + screening_failures
(RUN_ROOT / "funnel.json").write_text(
    json.dumps(funnel, indent=2), encoding="utf-8"
)
(RUN_ROOT / "mattergen-generation-funnels.json").write_text(
    json.dumps({
        "counts": search_generation_funnels,
        "hashes": search_generation_funnel_hashes,
        "global_exact_file_sha256s": sorted(raw_exact_file_hashes),
    }, indent=2),
    encoding="utf-8",
)
(RUN_ROOT / "crystal-groups.json").write_text(
    json.dumps({
        "matcher_settings": crystal_matcher_settings,
        "ambiguous_comparisons": crystal_ambiguous_comparisons,
        "groups": crystal_group_records,
    }, indent=2), encoding="utf-8"
)
(RUN_ROOT / "screening-records.json").write_text(
    json.dumps(screening_records, indent=2), encoding="utf-8"
)
(RUN_ROOT / "failures.json").write_text(
    json.dumps(failures, indent=2), encoding="utf-8"
)
(RUN_ROOT / "dft-handoff-report.json").write_text(
    dft_report.model_dump_json(indent=2), encoding="utf-8"
)
material_domain_final_audit = {
    **material_domain_audit,
    "material_domain_plan_relative_path": str(
        MATERIAL_DOMAIN_PLAN_PATH.relative_to(RUN_ROOT)
    ),
    "field_specific_property_calculation_executed": False,
    "required_field_properties_status": field_property_status,
    "stage_routes": {
        stage: run.report.domain_route.model_dump(mode="json")
        for stage, run in stage_evidence_runs.items()
    },
    "stage_evidence_statuses": {
        stage: str(run.report.status) for stage, run in stage_evidence_runs.items()
    },
    "archive_inclusion": "included-under-run-root",
}
MATERIAL_DOMAIN_FINAL_AUDIT_PATH = RUN_ROOT / "material-domain-final-audit.json"
MATERIAL_DOMAIN_FINAL_AUDIT_PATH.write_text(
    json.dumps(material_domain_final_audit, indent=2) + "\n",
    encoding="utf-8",
)
(RUN_ROOT / "stage-validation-evidence-index.json").write_text(
    json.dumps({
        "operator_skill": {
            "skill_id": PROJECT_SKILL_ID,
            "relative_path": str(PROJECT_SKILL_PATH.relative_to(REPOSITORY_DIR)),
            "sha256": PROJECT_SKILL_SHA256,
            "role": "operator-procedure-not-scientific-validator",
        },
        "material_domain": {
            "material_field": domain_plan.profile.material_field.value,
            "field_profile_id": domain_plan.profile.profile_id,
            "broad_validation_profile_id": selected_validation_profile.profile_id,
            "resolution_mode": domain_plan.resolution.selection_mode,
            "main_ai_field_routing_setting": MAIN_AI_FIELD_ROUTING,
            "main_ai_field_routing_status": main_ai_field_routing_status,
            "main_model_run": (
                domain_plan.main_model_run.model_dump(mode="json")
                if domain_plan.main_model_run is not None else None
            ),
            "plan_relative_path": str(MATERIAL_DOMAIN_PLAN_PATH.relative_to(RUN_ROOT)),
            "final_audit_relative_path": str(
                MATERIAL_DOMAIN_FINAL_AUDIT_PATH.relative_to(RUN_ROOT)
            ),
            "field_specific_property_calculation_executed": False,
            "unexecuted_required_properties": field_property_status,
            "scientific_boundary": GENERIC_T4_FIELD_PROPERTY_BOUNDARY,
        },
        "reports": stage_evidence_refs,
        "checkpoint_receipts": {
            stage: str(path.relative_to(RUN_ROOT))
            for stage, path in stage_checkpoint_receipt_paths.items()
        },
        "unknown_policy": STAGE_EVIDENCE_UNKNOWN_POLICY,
        "scientific_role": "search_and_validation_context_only",
        "property_scores_created": False,
    }, indent=2),
    encoding="utf-8",
)
run_summary = {
    "run_id": RUN_ID,
    "chemical_system": chemical_system,
    "material_domain": {
        "material_field": domain_plan.profile.material_field.value,
        "field_profile_id": domain_plan.profile.profile_id,
        "broad_discovery_domain": domain_plan.profile.discovery_domain.value,
        "broad_validation_profile_id": selected_validation_profile.profile_id,
        "resolution_mode": domain_plan.resolution.selection_mode,
        "main_ai_field_routing_setting": MAIN_AI_FIELD_ROUTING,
        "main_ai_field_routing_status": main_ai_field_routing_status,
        "main_model_run": (
            domain_plan.main_model_run.model_dump(mode="json")
            if domain_plan.main_model_run is not None else None
        ),
        "candidate_type": "crystal",
        "plan_relative_path": str(MATERIAL_DOMAIN_PLAN_PATH.relative_to(RUN_ROOT)),
        "final_audit_relative_path": str(
            MATERIAL_DOMAIN_FINAL_AUDIT_PATH.relative_to(RUN_ROOT)
        ),
        "field_specific_property_calculation_executed": False,
        "unexecuted_required_properties": field_property_status,
        "scientific_boundary": GENERIC_T4_FIELD_PROPERTY_BOUNDARY,
    },
    "search_plan": SEARCH_PLAN,
    "fusion_search": {
        "status": str(search_report.status),
        "rounds_completed": search_report.rounds_completed,
        "budget_usage": search_report.budget_usage.model_dump(mode="json"),
        "report_relative_path": str(FUSION_SEARCH_REPORT_PATH.relative_to(RUN_ROOT)),
    },
    "returned_authoritative_candidate_count": returned_candidate_count,
    "raw_audit_cifs_are_not_ranked": True,
    "final_unique_cif_directory": str(UNIQUE_CIF_ROOT.relative_to(RUN_ROOT)),
    "funnel": funnel,
    "composition_replication_audit": composition_replication_audit,
    "generation_profile_matrix": str(
        GENERATION_PROFILE_MATRIX_PATH.relative_to(RUN_ROOT)
    ),
    "novelty_dft_portfolio_receipt": str(
        NOVELTY_PORTFOLIO_PATH.relative_to(RUN_ROOT)
    ),
    "dft_package_count": len(dft_report.packages),
    "dft_calculation_executed": dft_report.calculation_executed,
    "scientific_boundary": dft_report.scientific_boundary,
    "stage_validation_evidence": stage_evidence_refs,
    "stage_checkpoint_receipts": {
        stage: str(path.relative_to(RUN_ROOT))
        for stage, path in stage_checkpoint_receipt_paths.items()
    },
    "operator_skill_id": PROJECT_SKILL_ID,
    "operator_skill_sha256": PROJECT_SKILL_SHA256,
    "stage_evidence_unknown_policy": STAGE_EVIDENCE_UNKNOWN_POLICY,
    "stage_evidence_property_scores_created": False,
    "credentials_serialized": False,
}
(RUN_ROOT / "run-summary.json").write_text(
    json.dumps(run_summary, indent=2), encoding="utf-8"
)
print(json.dumps(run_summary, indent=2))


## Checks

The final cell enforces the exact funnel, the full requested cross-round crystallographic-unique count, multiple actually applied alpha/gamma profiles, at least three applied hull-target profiles, and non-increasing downstream counts. It validates all five request/report/handoff/receipt chains, the bounded external no-match DFT portfolio, the procedural skill hash and stage-specific MCP allowlists, preserves unknown/failure states, verifies both expert and relaxation records, confirms that no thermodynamic value was invented, and writes a portable ZIP without credentials.

In [ ]:
# @title 7. Enforce scientific boundaries and export the run
EXPECTED_FUNNEL_NAMES = [
    "requested_samples",
    "raw_model_structures",
    "parsed_structures",
    "exact_file_unique",
    "crystallographically_unique",
    "geometry_valid",
    "mlip_evaluated",
    "relaxation_converged",
    "ranked_candidates",
]
if list(funnel) != EXPECTED_FUNNEL_NAMES:
    raise AssertionError("Funnel names or order changed.")
if funnel["requested_samples"] != TOTAL_CANDIDATES:
    raise AssertionError("Requested samples must equal the allocated 8-32 batch.")
if returned_candidate_count != TOTAL_CANDIDATES:
    raise AssertionError("The global Fusion search did not fill the requested candidate budget.")
if funnel["crystallographically_unique"] != TOTAL_CANDIDATES:
    raise AssertionError(
        "The final cross-round crystallographically unique set must fill the requested budget."
    )
if search_report.budget_usage.generated_candidates != TOTAL_CANDIDATES:
    raise AssertionError("Persisted search budget usage differs from the requested total.")
if search_report.budget_usage.generation_calls > MAX_GENERATION_CALLS:
    raise AssertionError("The global generation-call budget was exceeded.")
if funnel["raw_model_structures"] < funnel["parsed_structures"]:
    raise AssertionError("Raw model structures cannot be below parsed structures.")
if funnel["parsed_structures"] < funnel["exact_file_unique"]:
    raise AssertionError("Parsed raw structures cannot be below global returned exact-file uniques.")
downstream_names = [
    "exact_file_unique", "crystallographically_unique", "geometry_valid",
    "mlip_evaluated", "relaxation_converged", "ranked_candidates",
]
downstream_values = [funnel[name] for name in downstream_names]
if any(left < right for left, right in zip(downstream_values, downstream_values[1:])):
    raise AssertionError("The returned-candidate screening funnel must be non-increasing.")
if search_report.rounds_requested != SEARCH_ROUNDS:
    raise AssertionError("Persisted search rounds differ from the user setting.")
if search_report.rounds_completed < 3:
    raise AssertionError("A real search must complete at least three adaptive rounds.")
EXPECTED_SEARCH_BRANCHES = {
    "pareto", "stability", "target_property", "novelty", "expert_disagreement"
}
if {str(branch.branch) for branch in search_report.branches} != EXPECTED_SEARCH_BRANCHES:
    raise AssertionError("The search must preserve all five exploration branches.")
if not any(branch.scheduler_history for branch in search_report.branches):
    raise AssertionError("The adaptive scheduler did not record control history.")
if len(applied_alphas) < 2 or len(applied_gammas) < 2:
    raise AssertionError("At least two alpha/gamma profiles must be applied, not only requested.")
if len(applied_target_profiles) < 3:
    raise AssertionError("At least three hull-target profiles must be applied.")
if not GENERATION_PROFILE_MATRIX_PATH.is_file():
    raise AssertionError("The applied generation-profile matrix was not persisted.")
EXPECTED_EVIDENCE_STAGES = {
    "generation_prior",
    "identity_novelty",
    "mlip_disagreement",
    "relaxation_validation",
    "dft_handoff",
}
if set(stage_evidence_runs) != EXPECTED_EVIDENCE_STAGES:
    raise AssertionError("All five stage-specific evidence routes are required.")
if set(stage_checkpoint_receipts) != EXPECTED_EVIDENCE_STAGES:
    raise AssertionError("Every evidence route requires an executable checkpoint receipt.")
if not MATERIAL_DOMAIN_PLAN_PATH.is_file() or not MATERIAL_DOMAIN_FINAL_AUDIT_PATH.is_file():
    raise AssertionError("The material-domain plan and final audit must be persisted.")
persisted_material_domain_plan = json.loads(
    MATERIAL_DOMAIN_PLAN_PATH.read_text(encoding="utf-8")
)
if persisted_material_domain_plan != domain_plan.model_dump(mode="json"):
    raise AssertionError("The full audited material-domain plan changed on disk.")
if MAIN_AI_FIELD_ROUTING == "REQUIRED" and domain_plan.main_model_run is None:
    raise AssertionError("REQUIRED main-AI routing completed without a model decision.")
if main_ai_field_routing_status == "executed-audited-decision":
    if domain_plan.main_model_run is None:
        raise AssertionError("Executed main-AI routing is missing its audited model run.")
    if domain_plan.main_model_run.endpoint_or_tool_selection_performed:
        raise AssertionError("The main field classifier must not choose endpoints or MCP tools.")
    if domain_plan.resolution.model_decision_id != domain_plan.main_model_run.decision_id:
        raise AssertionError("The reconciled route does not cite its model decision.")
if domain_plan.resolution.requires_operator_choice:
    raise AssertionError("An ambiguous AUTO material field must stop before generation.")
if search_goal.domain != domain_plan.profile.discovery_domain:
    raise AssertionError("The DiscoveryGoal did not use the selected broad domain.")
if search_goal.validation_profile_id != selected_validation_profile.profile_id:
    raise AssertionError("The DiscoveryGoal validation profile does not match its domain.")
if search_goal.candidate_types != [CandidateType.CRYSTAL]:
    raise AssertionError("The generic T4 workflow must remain crystal-only.")
if set(field_property_status) != set(domain_plan.unexecuted_required_properties):
    raise AssertionError("Every unexecuted required field property must remain explicit.")
if any(
    item["status"] != "unknown"
    or item["execution"] != "not_executed"
    or item["unknown_is_pass"]
    for item in field_property_status.values()
):
    raise AssertionError("Generic MLIP triage cannot pass a field-specific property.")
if hashlib.sha256(PROJECT_SKILL_PATH.read_bytes()).hexdigest() != PROJECT_SKILL_SHA256:
    raise AssertionError("The material-candidate-validation operator skill changed during the run.")
for stage, run in stage_evidence_runs.items():
    report = run.report
    stage_enum = ValidationEvidenceStage(stage)
    contract = STAGE_EXECUTION_CONTRACTS[stage_enum]
    receipt = stage_checkpoint_receipts[stage]
    receipt_path = stage_checkpoint_receipt_paths[stage]
    if not receipt_path.is_file():
        raise AssertionError(stage + " checkpoint receipt was not persisted.")
    if report.material_field != domain_plan.profile.material_field:
        raise AssertionError(stage + " did not preserve the selected material field.")
    if report.domain_route is None or report.domain_route.profile_id != domain_plan.profile.profile_id:
        raise AssertionError(stage + " is missing its code-owned material-field route.")
    if report.property_score_created or report.handoff.property_score_created:
        raise AssertionError(stage + " evidence must not create property scores.")
    if not report.handoff.unknown_not_pass:
        raise AssertionError(stage + " evidence must preserve unknown-not-pass.")
    if str(report.status) in {"unknown", "skipped"}:
        if not report.reason or report.handoff.evidence_available:
            raise AssertionError(stage + " unknown/skipped evidence must stay unavailable.")
    if str(report.status) in {"completed", "partial"} and not report.handoff.evidence_available:
        raise AssertionError(stage + " available evidence requires a typed handoff.")
    if list(report.route.official_validators) != contract["official_validators"]:
        raise AssertionError(stage + " official validator allowlist changed.")
    if [str(item) for item in report.route.literature_sources] != contract[
        "allowed_literature_sources"
    ]:
        raise AssertionError(stage + " scholarly source allowlist changed.")
    if report.route.mcp_policy != "configured-tool-only":
        raise AssertionError("Only the administrator-configured MCP tool is allowed.")
    if report.route.mcp_contract.selection_policy != (
        "administrator-configured-allowlist-only"
    ):
        raise AssertionError("MCP selection must not come from a model or prompt.")
    if receipt["allowlist"]["model_or_prompt_can_select_tool"]:
        raise AssertionError("Checkpoint receipt weakened the MCP selection boundary.")
    if receipt["status_handling"]["unknown_is_pass"]:
        raise AssertionError("Unknown evidence cannot be recorded as passing.")
    if set(receipt["runtime_validator"]["authority_outcomes"]) != set(
        contract["official_validators"]
    ):
        raise AssertionError(stage + " runtime authority provenance is incomplete.")
    if any(
        value not in VALID_RUNTIME_AUTHORITY_STATES
        for value in receipt["runtime_validator"]["authority_outcomes"].values()
    ):
        raise AssertionError(stage + " runtime authority state is invalid.")
    if stage == "generation_prior":
        if report.handoff.can_steer_generation and not report.handoff.evidence_branch_ids:
            raise AssertionError("Generation steering requires a source-closed branch.")
    elif report.handoff.can_steer_generation or report.handoff.evidence_branch_ids:
        raise AssertionError("Post-generation evidence cannot become generator control.")
    configured_tool = (
        os.environ.get(report.route.mcp_contract.tool_environment_variable, "").strip()
        or os.environ.get(
            report.route.mcp_contract.fallback_tool_environment_variable, ""
        ).strip()
        or None
    )
    if str(report.mcp_contract_status) == "verified":
        if report.mcp_tool_name != configured_tool:
            raise AssertionError(stage + " MCP tool provenance differs from its allowlist.")
    elif report.mcp_tool_name is not None:
        raise AssertionError(stage + " unverified MCP tool cannot enter provenance.")
for cycle in search_report.cycle_records:
    if cycle.evidence_branch_id is not None and not cycle.evidence_claim_ids:
        raise AssertionError("Evidence-guided generation must stay claim-closed.")
if search_report.search_budget is None:
    raise AssertionError("A global search budget must be persisted.")
if search_report.search_budget.max_generated_candidates != TOTAL_CANDIDATES:
    raise AssertionError("Persisted candidate budget changed.")
if search_report.search_budget.max_generation_calls != MAX_GENERATION_CALLS:
    raise AssertionError("Persisted generation-call budget changed.")

if composition_replication_audit["eligible_relaxed_candidate_count"] != len(
    relaxation_converged_rows
):
    raise AssertionError("Composition replication denominator changed.")
replication_fraction = composition_replication_audit["singleton_composition_fraction"]
if replication_fraction is not None and replication_fraction > (
    SINGLETON_COMPOSITION_FRACTION_THRESHOLD
) and composition_replication_audit["conclusion"] != (
    "insufficient_within_composition_replication"
):
    raise AssertionError("High singleton coverage must be labelled insufficient.")
if composition_replication_audit["global_priority_scope"] != (
    "triage_by_force_diversity_uncertainty_not_cross_formula_stability_ranking"
):
    raise AssertionError("Global priority must not imply cross-formula stability.")
if composition_replication_audit["rag_numeric_stability_substitution_performed"]:
    raise AssertionError("RAG evidence cannot substitute a numeric stability value.")

for record in screening_records:
    if set(record["screening_predictions"]["raw_feature_artifacts"]) != {
        "mattersim", "chgnet"
    }:
        raise AssertionError("Both original feature artifacts must be retained.")
    relaxation = record["relaxation_validation"]
    if set(relaxation).issuperset({"mattersim", "chgnet"}) is False:
        raise AssertionError("Both independent relaxation records are required.")
    if any(
        relaxation[expert]["endpoint_operation"] != "/v1/relax"
        for expert in ("mattersim", "chgnet")
    ):
        raise AssertionError("Relaxations must be direct /v1/relax operations.")
    if record["generation_conditions"]["scientific_role"] != (
        "generation_target_not_a_measured_property"
    ):
        raise AssertionError("Generation conditions crossed the evidence boundary.")
    if record["screening_predictions"][
        "cross_stoichiometry_energy_comparison_performed"
    ]:
        raise AssertionError("Absolute MLIP energies cannot be compared across stoichiometries.")
    thermodynamics = record["thermodynamic_validation"]
    if thermodynamics["status"] != "not_run" or thermodynamics["dft_executed"]:
        raise AssertionError("DFT execution was falsely reported.")
    for field in (
        "formation_energy_eV_atom",
        "computed_energy_above_hull_eV_atom",
        "reference_phase_set",
        "method",
    ):
        if thermodynamics[field] is not None:
            raise AssertionError("Uncalculated thermodynamic fields must remain null.")
    if record["authoritative_candidate_cif"]["raw_audit_cif_counted_or_ranked"]:
        raise AssertionError("Raw audit CIFs must not enter screening or ranking.")
    if record["novelty"]["project_history"]["status"] != "unknown":
        raise AssertionError("Unconfigured project history must remain unknown.")
    predictions = record["screening_predictions"]
    relaxed_predictions = predictions["relaxed_ranking_predictions"]
    if predictions["entered_final_ranked_funnel"]:
        if relaxed_predictions is None or predictions["composition_scoped_pareto"] is None:
            raise AssertionError("Only verified relaxed predictions may enter Pareto ranking.")
    elif predictions["composition_scoped_pareto"] is not None:
        raise AssertionError("Unranked relaxation cannot carry a Pareto result.")
    disagreement = predictions["model_disagreement"]
    relative_energy = predictions["composition_relative_energy_disagreement"]
    geometry_basis = predictions["disagreement_evidence_geometry"]
    expected_common_geometry_alignment_id = stable_hash({
        "schema": "common-geometry-mlip-alignment-v1",
        "candidate_ref": record["candidate_ref"],
        "force_alignment_audit": predictions["force_alignment_audit"],
        "feature_artifacts": predictions["raw_feature_artifacts"],
    })
    if predictions["initial_common_geometry_alignment_id"] != (
        expected_common_geometry_alignment_id
    ):
        raise AssertionError("Common-geometry alignment receipt changed.")
    if predictions["initial_common_geometry_force_rmse_eV_A"] != disagreement["force_rmse_eV_A"]:
        raise AssertionError("Force RMSE must remain bound to the initial common geometry.")
    if predictions["initial_common_geometry_stress_diff_GPa"] != disagreement["stress_norm_abs_diff_GPa"]:
        raise AssertionError("Stress difference must remain bound to the initial common geometry.")
    if geometry_basis["raw_energy_force_stress"] != "initial_common_candidate_geometry":
        raise AssertionError("Cross-model force and stress require one common geometry.")
    if geometry_basis["energy_risk"] != "composition_relative_independently_relaxed_geometries":
        raise AssertionError("Energy risk must use the independently relaxed composition panel.")
    if geometry_basis["relaxed_force_cross_model_comparison_performed"]:
        raise AssertionError("Relaxed forces from different geometries cannot be compared.")
    if geometry_basis["relaxed_stress_cross_model_comparison_performed"]:
        raise AssertionError("Relaxed stresses from different geometries cannot be compared.")
    if disagreement["energy_comparison_basis"] == "unknown":
        if relative_energy["status"] != "unknown" or not disagreement["dft_escalation"]:
            raise AssertionError("Unknown composition-relative energy must escalate to DFT.")
    elif relative_energy["status"] != "available":
        raise AssertionError("Aligned energy risk requires available relative evidence.")
    if disagreement["risk"] == "high" and not record["dft_escalation"][
        "required_due_to_high_disagreement"
    ]:
        raise AssertionError("High disagreement must enter the DFT escalation queue.")

if dft_report.calculation_executed or not 1 <= len(dft_report.packages) <= 5:
    raise AssertionError("DFT handoff must contain 1-5 non-executed input packages.")
if any(package.manifest.calculation_executed for package in dft_report.packages):
    raise AssertionError("An input package cannot claim a completed DFT calculation.")
if not NOVELTY_PORTFOLIO_PATH.is_file() or novelty_portfolio.max_novelty_slots != 1:
    raise AssertionError("The bounded novelty-portfolio receipt is missing or invalid.")
reserved_external_ref = novelty_portfolio.reserved_external_no_match_ref
if reserved_external_ref is not None:
    reserved_key = ref_key(reserved_external_ref)
    if reserved_key not in selected_dft_keys:
        raise AssertionError("The reserved external no-match candidate was not packaged.")
    if str(novelty_by_ref[reserved_key].external_database.status) != "no_match":
        raise AssertionError("Only a strict external-database no-match may use the slot.")

for secret_name, secret in credential_env_values.items():
    for path in RUN_ROOT.rglob("*"):
        if path.is_file() and secret.encode() in path.read_bytes():
            raise AssertionError(
                secret_name + " was serialized into " + str(path)
            )
    del secret

archive_path = Path(
    shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
)
print(json.dumps({
    "checks": "passed",
    "material_field": domain_plan.profile.material_field.value,
    "material_field_resolution_mode": domain_plan.resolution.selection_mode,
    "main_ai_field_routing_status": main_ai_field_routing_status,
    "main_model_decision_id": (
        domain_plan.main_model_run.decision_id
        if domain_plan.main_model_run is not None else None
    ),
    "unexecuted_required_field_properties": domain_plan.unexecuted_required_properties,
    "funnel": funnel,
    "dft_packages": len(dft_report.packages),
    "material_domain_plan": str(MATERIAL_DOMAIN_PLAN_PATH),
    "material_domain_final_audit": str(MATERIAL_DOMAIN_FINAL_AUDIT_PATH),
    "archive": str(archive_path),
}, indent=2))

if DOWNLOAD_RESULTS_ZIP:
    try:
        from google.colab import files
        files.download(str(archive_path))
    except ImportError:
        print("Not running in Colab; archive remains at", archive_path)


## Next Steps

Review every DFT manifest, choose licensed pseudopotentials, replace all placeholders, and run convergence-tested relax/static calculations outside this notebook. Only then compute reference-consistent formation energies and a phase diagram. Experimental synthesis and characterization remain required before making a discovery claim.